# Collision Anticipation: Model

<div dir="rtl">

**هدف.** مدلی که در هر لحظه فقط ۲ ثانیه‌ی آخر ویدئو را ببیند و یک امتیاز خطر بدهد: آیا تا ۰.۵ تا ۱.۵ ثانیه‌ی دیگر تصادف رخ می‌دهد؟

**داده.** ویژگی‌های DINOv2 از نوت‌بوک پیش‌پردازش (۱۰ فریم در ثانیه و برای هر فریم ۵ بردار ۳۸۴تایی) و جدول `videos.csv`. برچسب پنجره‌های train از `time_of_alert` و `time_of_event` ساخته می‌شود.

**مراحل.**
1. داده: بار کردن ویژگی‌ها، بررسی ویدئوهای تکراری، تغییر فریم‌ها پیش از حادثه، پنجره‌ها و برچسب‌ها، تقسیم ویدئوها به پنج دسته.
2. مقایسه‌ی مدل‌ها با cross-validation.
3. تحلیل مدل انتخاب‌شده.
4. مدل نهایی و ارزیابی روی آزمون رسمی Nexar.

**فایل‌های لازم.** پوشه‌های `features/` و `data/` در `MyDrive/project2/anticipation/` که نوت‌بوک پیش‌پردازش ساخته است. خروجی‌ها در `results/` و `models/` همان پوشه ذخیره می‌شوند.

</div>

## Setup

<div dir="rtl">

از `videos.csv` برای ویدئوهای train زمان هشدار و حادثه خوانده می‌شود. از آزمون فقط `time_to_accident` استفاده می‌شود، که گروه Δ هر کلیپ است و اسکریپت رسمی هم با همان گروه‌بندی می‌کند. متادیتای آزمون زمان حادثه‌ی کلیپ‌های مثبت را هم دارد، ولی این ستون‌ها هیچ‌جا به کار نمی‌روند. ورودی مدل فقط ویژگی فریم‌هاست.

</div>

In [1]:
from itertools import combinations
from pathlib import Path
import time
import warnings

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.decomposition import PCA
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold

pio.templates.default = "plotly_white"
warnings.filterwarnings("ignore", message=".*eval_set.*")  # newer LightGBM versions rename eval_set
warnings.filterwarnings("ignore", message="X does not have valid feature names")

In [2]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = Path("/content/drive/MyDrive/project2/anticipation")
FEATURE_DIR = DRIVE_DIR / "features"
DATA_DIR = DRIVE_DIR / "data"
RESULT_DIR = DRIVE_DIR / "results"
RESULT_DIR.mkdir(exist_ok=True)

WINDOW_FRAMES = 20    # 2 s at 10 fps
LONGEST_WINDOW = 30   # 3 s, the longest window compared later
STEP = 0.2            # spacing of risk windows (s)
RISK_END = 0.3        # risk windows end at least this long before the event
BUFFER = 1.0          # unlabeled second before the alert
HARD_SPAN = 8.0       # hard negatives end within this span before the buffer
N_HARD = 6            # hard negatives per collision video
N_NORMAL = 12         # windows per normal-driving video
DELTAS = [0.5, 1.0, 1.5]
N_FOLDS = 5
SEED = 42

Mounted at /content/drive


In [3]:
videos = pd.read_csv(DATA_DIR / "videos.csv", dtype={"video_id": str})
videos["gap"] = videos["time_of_event"] - videos["time_of_alert"]
print("Videos (columns: label):")
print(pd.crosstab(videos["split"], videos["label"], margins=True))

test = videos[videos["split"] != "train"]
with_event = test.groupby("label")["time_of_event"].count().to_dict()
print("\nTest clips with a time_of_event value (by label):", with_event)

Videos (columns: label):
label            0     1   All
split                         
test-private   339   338   677
test-public    333   334   667
train          750   750  1500
All           1422  1422  2844

Test clips with a time_of_event value (by label): {0: 0, 1: 672}


## Frame Features

<div dir="rtl">

ویژگی همه‌ی فریم‌ها در یک آرایه کنار هم قرار می‌گیرد و برای هر ویدئو نگه داشته می‌شود که فریم‌هایش از کجای آرایه شروع می‌شوند. ۵ بردار هر فریم (CLS و چهار ربع تصویر) پشت هم قرار می‌گیرند و یک بردار ۱۹۲۰تایی می‌سازند. ۳۸۴ عدد اول همان CLS است.

</div>

In [4]:
feature_parts, time_parts, index_rows = [], [], []
start = 0
for path in sorted(FEATURE_DIR.glob("*/chunk_*.npz")):
    with np.load(path) as chunk:
        n = len(chunk["times"])
        feature_parts.append(chunk["features"].reshape(n, -1))
        time_parts.append(chunk["times"])
        offsets = chunk["offsets"]
        for i, video_path in enumerate(chunk["video_paths"]):
            index_rows.append((str(video_path), start + offsets[i], offsets[i + 1] - offsets[i]))
    start += n
FEATS = np.concatenate(feature_parts)
TIMES = np.concatenate(time_parts)
del feature_parts, time_parts

frame_index = pd.DataFrame(index_rows, columns=["video_path", "start", "frames"])
videos = videos.merge(frame_index, on="video_path", how="left")
VIDEO_FRAMES = dict(zip(videos["video_path"], zip(videos["start"], videos["frames"])))

print(f"Frames in memory: {FEATS.shape} | {FEATS.nbytes / 1e9:.2f} GB")
print("Videos with features:", videos["start"].notna().sum(), "of", len(videos))
print("Frame counts equal to videos.csv:", (videos["frames"] == videos["n_frames"]).all())

Frames in memory: (570322, 1920) | 2.19 GB
Videos with features: 2844 of 2844
Frame counts equal to videos.csv: True


<div dir="rtl">

**نتیجه.**
- هر ۲۸۴۴ ویدئو ویژگی دارند: ۵۷۰٬۳۲۲ فریم و ۲.۱۹ گیگ در حافظه. تعداد فریم هر ویدئو با `videos.csv` یکی است.
- در آزمون، هر ۶۷۲ کلیپ مثبت `time_of_event` دارند و هیچ کلیپ منفی‌ای ندارد. پس این ستون به‌تنهایی برچسب را لو می‌دهد، و به همین دلیل در هیچ مرحله‌ای استفاده نمی‌شود.

</div>

## Duplicate Videos

<div dir="rtl">

تقسیم داده بر اساس ویدئوست. اگر یک صحنه‌ی یکسان در دو ویدئو باشد، ممکن است یکی در آموزش و دیگری در ارزیابی قرار بگیرد و نتیجه خوش‌بینانه شود. برای هر ویدئو میانگین بردار CLS در هر ثانیه حساب می‌شود، و شباهت دو ویدئو شباهت کسینوسی شبیه‌ترین ثانیه‌های آن‌هاست.

برای اینکه معلوم باشد صحنه‌ی یکسان چه شباهتی می‌دهد، از خود آزمون کمک گرفته می‌شود. هر ویدئوی آزمون تا سه کلیپ دارد (یکی برای هر Δ) و `time_to_accident_test_map.csv` می‌گوید کدام کلیپ‌ها از یک ویدئو بریده شده‌اند. مرز «صحنه‌ی یکسان» شباهتی است که ۹۹٪ این جفت‌ها از آن بالاترند.

</div>

In [5]:
def second_vectors(video_path):
    s, n = VIDEO_FRAMES[video_path]
    cls = FEATS[s:s + n, :384].astype(np.float32)
    k = n // 10
    seconds = cls[:k * 10].reshape(k, 10, 384).mean(axis=1)
    return seconds / np.linalg.norm(seconds, axis=1, keepdims=True)


def nearest_video(query_rows, base_rows):
    # for each query video: the other base video with the most similar second
    base = np.concatenate([SECONDS[i] for i in base_rows])
    starts = np.cumsum([0] + [len(SECONDS[i]) for i in base_rows[:-1]])
    result = []
    for i in query_rows:
        per_video = np.maximum.reduceat((SECONDS[i] @ base.T).max(axis=0), starts)
        per_video[base_rows == i] = -1
        j = per_video.argmax()
        nearest = videos.at[base_rows[j], "video_path"]
        result.append((videos.at[i, "video_path"], nearest, per_video[j]))
    return pd.DataFrame(result, columns=["video_path", "nearest", "similarity"])


SECONDS = [second_vectors(p) for p in videos["video_path"]]

test_map = pd.read_csv(DATA_DIR / "time_to_accident_test_map.csv")
row_of = {int(v): i for i, v in videos.loc[videos["split"] != "train", "video_id"].items()}
control = []
for ids in test_map.itertuples(index=False):
    rows = [row_of[int(x)] for x in ids if pd.notna(x)]
    for a, b in combinations(rows, 2):
        control.append((SECONDS[a] @ SECONDS[b].T).max())
control = np.array(control)
SAME = np.quantile(control, 0.01)

train_rows = np.flatnonzero(videos["split"] == "train")
test_rows = np.flatnonzero(videos["split"] != "train")
within_train = nearest_video(train_rows, train_rows)
test_to_train = nearest_video(test_rows, train_rows)
duplicates = within_train[within_train["similarity"] >= SAME]

print(f"Clips of the same test video: {len(control)} pairs | similarity min {control.min():.3f} | "
      f"1st percentile {SAME:.3f} | median {np.median(control):.3f}")
for name, table in [("Train video vs nearest other train video", within_train),
                    ("Test clip vs nearest train video", test_to_train)]:
    sim = table["similarity"]
    print(f"{name}: median {sim.median():.3f} | max {sim.max():.3f} | "
          f"at or above {SAME:.3f}: {(sim >= SAME).sum()}")

pairs = pd.concat([pd.DataFrame({"similarity": control, "pair": "clips of the same test video"}),
                   within_train.assign(pair="train video vs nearest other train video"),
                   test_to_train.assign(pair="test clip vs nearest train video")])
fig = px.histogram(pairs, x="similarity", color="pair", barmode="overlay", nbins=80,
                   histnorm="percent", opacity=0.6, title="Similarity of the best-matching second")
fig.add_vline(x=SAME, line_dash="dash", annotation_text="same-video 1st percentile")
fig.update_layout(height=400, yaxis_title="percent of pairs")
fig.show()
flagged = pd.concat([duplicates, test_to_train[test_to_train["similarity"] >= SAME]])
flagged.sort_values("similarity", ascending=False).head(10).round(3)

Clips of the same test video: 1088 pairs | similarity min 0.982 | 1st percentile 0.987 | median 0.997
Train video vs nearest other train video: median 0.867 | max 0.996 | at or above 0.987: 2
Test clip vs nearest train video: median 0.855 | max 0.946 | at or above 0.987: 0


,video_path,nearest,similarity
435,train/positive/00788.mp4,train/positive/00789.mp4,0.996
485,train/positive/00789.mp4,train/positive/00788.mp4,0.996


<div dir="rtl">

**نتیجه.**
- ۱۰۸۸ جفت کلیپ از یک ویدئوی آزمون شباهتی بین ۰.۹۸۲ و ۱ دارند (میانه ۰.۹۹۷). مرز «صحنه‌ی یکسان» صدک ۱ این جفت‌هاست: ۰.۹۸۷.
- شبیه‌ترین ویدئوی train به هر ویدئوی train دیگر، به‌طور میانه ۰.۸۶۷ شباهت دارد. فقط یک جفت از مرز گذشت: ویدئوهای تصادف `00788` و `00789`، با شماره‌ی پشت‌سرهم و شباهت ۰.۹۹۶. این دو در بخش Folds در یک دسته قرار می‌گیرند.
- بیشترین شباهت یک کلیپ آزمون به ویدئوهای train ۰.۹۴۶ است، کمتر از کمترین شباهت کلیپ‌های یک ویدئو (۰.۹۸۲). پس صحنه‌ی مشترکی بین آزمون و train پیدا نشد.

</div>

## Frame Change Before the Event

<div dir="rtl">

برای هر فریم، تغییر بردار CLS نسبت به فریم قبلی (یک منهای شباهت کسینوسی) حساب و بر حسب فاصله تا حادثه میانگین گرفته می‌شود. ویدئوهای عادی حادثه ندارند، پس به هر کدام یک زمان حادثه‌ی ساختگی داده می‌شود که به تصادف از زمان حادثه‌های واقعی train برداشته شده است. همین زمان‌ها بعداً جای پنجره‌های ارزیابی را هم تعیین می‌کنند. این بخش توصیفی است و فقط از train استفاده می‌کند.

</div>

In [6]:
rng = np.random.default_rng(SEED)
train = videos[videos["split"] == "train"].reset_index(drop=True)
events = train.loc[train["label"] == 1, "time_of_event"].to_numpy()
normal = train["label"] == 0
train.loc[normal, "fake_event"] = rng.choice(events, normal.sum())
train["event"] = train["time_of_event"].fillna(train["fake_event"])

used = train[(train["event"] >= 8) & (train["last_time"] >= train["event"] + 1.9)]
print("Videos with 8 s before and 2 s after the event:", used["label"].value_counts().to_dict())

parts = []
for v in used.itertuples():
    s, n = VIDEO_FRAMES[v.video_path]
    cls = FEATS[s:s + n, :384].astype(np.float32)
    cls /= np.linalg.norm(cls, axis=1, keepdims=True)
    rel = TIMES[s + 1:s + n] - v.event
    keep = (rel >= -8) & (rel <= 2)
    parts.append(pd.DataFrame({"video": "collision" if v.label else "normal driving",
                               "time": (np.round(rel[keep] / 0.2) * 0.2).round(1),
                               "change": 1 - (cls[1:] * cls[:-1]).sum(axis=1)[keep]}))
change = pd.concat(parts)
periods = ["8 to 3 s before", "3 to 1.5 s before", "last 1.5 s", "after the event"]
change["period"] = pd.cut(change["time"], [-8.1, -3, -1.5, 0, 2.1], labels=periods)
print("\nMean frame-to-frame change:")
print(change.groupby(["video", "period"], observed=True)["change"].mean().unstack().round(4))

curve = change.groupby(["video", "time"])["change"].mean().reset_index()
fig = px.line(curve, x="time", y="change", color="video",
              labels={"time": "seconds from the event (fake event for normal driving)",
                      "change": "1 - cosine similarity"},
              title="Mean frame-to-frame change of the CLS vector (train)")
fig.add_vline(x=0, line_dash="dash", line_color="red", annotation_text="event")
fig.add_vline(x=-train["gap"].median(), line_dash="dash", line_color="orange",
              annotation_text="median alert")
fig.update_layout(height=400)
fig.show()

Videos with 8 s before and 2 s after the event: {1: 739, 0: 650}

Mean frame-to-frame change:
period          8 to 3 s before  3 to 1.5 s before  last 1.5 s  \
video                                                            
collision                0.0256             0.0280      0.0347   
normal driving           0.0304             0.0318      0.0327   

period          after the event  
video                            
collision                0.0313  
normal driving           0.0306  


<div dir="rtl">

**نتیجه** (۷۳۹ ویدئوی تصادف و ۶۵۰ ویدئوی عادی که ۸ ثانیه قبل و ۲ ثانیه بعد از حادثه را دارند).
- در ویدئوهای تصادف، تغییر فریم‌ها با نزدیک شدن به حادثه زیاد می‌شود: ۰.۰۲۵۶ در ۸ تا ۳ ثانیه قبل، ۰.۰۲۸۰ در ۳ تا ۱.۵ ثانیه قبل و ۰.۰۳۴۷ در ۱.۵ ثانیه‌ی آخر، یعنی حدود ۳۶٪ بیشتر از ابتدا. در نمودار، بیشترین مقدار درست بعد از حادثه است.
- در رانندگی عادی همین عدد تقریباً ثابت است: بین ۰.۰۳۰۴ و ۰.۰۳۲۷.
- سطح پایه‌ی ویدئوهای عادی بالاتر است (۰.۰۳۰۴ در برابر ۰.۰۲۵۶). حتی ۳ تا ۱.۵ ثانیه قبل از حادثه، تغییر در ویدئوهای تصادف (۰.۰۲۸۰) کمتر از رانندگی عادی (۰.۰۳۱۸) است. پس مقدار تغییر به‌تنهایی دو دسته را جدا نمی‌کند، و چیزی که فرق دارد افزایش آن نسبت به ثانیه‌های قبلِ همان ویدئوست. این یکی از دلیل‌های ساختن منفی‌های سخت از خود ویدئوهای تصادف است.
- افزایش از بازه‌ی ۳ تا ۱.۵ ثانیه قبل از حادثه دیده می‌شود. یک پنجره‌ی ۲ ثانیه‌ای که ۱.۵ ثانیه قبل از حادثه تمام شود، همین بازه را می‌بیند.

</div>

## Training Windows

<div dir="rtl">

هر نمونه یک پنجره‌ی ۲ ثانیه‌ای (۲۰ فریم) است که در لحظه‌ی `t` تمام می‌شود.
- **خطر (۱):** در ویدئوی تصادف، `t` از `time_of_alert` تا ۰.۳ ثانیه قبل از حادثه، هر ۰.۲ ثانیه.
- **منفی سخت (۰):** ۶ پنجره از همان ویدئو، پیش از `alert − 1`. صحنه همان صحنه‌ی پنجره‌های خطر است، پس نوع جاده سرنخ نمی‌شود.
- **رانندگی عادی (۰):** ۱۲ پنجره از هر ویدئوی عادی: نصف کمی قبل از یک زمان حادثه‌ی واقعی که در طول ویدئو جا شود، نصف از هر جای ویدئو.

ثانیه‌ی قبل از `time_of_alert` برچسب نمی‌گیرد، چون طبق مقاله‌ی دیتاست این زمان دومین زمانِ دیرتر بین ۱۰ برچسب‌زن است.

</div>

In [7]:
rows = []
for v in train.itertuples():
    if v.label == 1:
        for t in np.arange(v.time_of_alert, v.time_of_event - RISK_END + 1e-6, STEP):
            rows.append((v.video_path, t, 1, "risk"))
        hard_end = v.time_of_alert - BUFFER
        if hard_end > 2:
            for t in rng.uniform(max(2, hard_end - HARD_SPAN), hard_end, N_HARD):
                rows.append((v.video_path, t, 0, "hard negative"))
    else:
        fitting = events[events <= v.last_time]
        matched = rng.choice(fitting, N_NORMAL // 2) - rng.uniform(RISK_END, 2, N_NORMAL // 2)
        anywhere = rng.uniform(2, v.last_time, N_NORMAL // 2)
        for t in np.concatenate([matched, anywhere]):
            rows.append((v.video_path, t, 0, "normal driving"))
windows = pd.DataFrame(rows, columns=["video_path", "t_end", "label", "kind"])


def end_frame(table, length=WINDOW_FRAMES):
    # index in FEATS of the last frame at or before t_end; -1 if fewer than `length` frames up to it
    result = np.full(len(table), -1)
    for i, (video_path, t) in enumerate(zip(table["video_path"], table["t_end"])):
        s, n = VIDEO_FRAMES[video_path]
        k = np.searchsorted(TIMES[s:s + n], t + 1e-4) - 1
        if k >= length - 1:
            result[i] = s + k
    return result


windows["end"] = end_frame(windows)
windows = windows[windows["end"] >= 0].reset_index(drop=True)
print(windows.groupby("kind").agg(windows=("label", "size"), videos=("video_path", "nunique")))

                windows  videos
kind                           
hard negative      4494     749
normal driving     8993     750
risk               5295     726


In [8]:
v = train[(train["label"] == 1) & train["gap"].between(1.5, 2.5)].iloc[0]
part = windows[windows["video_path"] == v["video_path"]]
fig = px.scatter(part, x="t_end", y="kind", color="kind",
                 labels={"t_end": "window end time (s)", "kind": ""},
                 title=f"Window labels in collision video {v['video_id']}")
fig.add_vrect(x0=v["time_of_alert"] - BUFFER, x1=v["time_of_alert"], fillcolor="lightgray",
              opacity=0.5, line_width=0, annotation_text="buffer", annotation_position="top left")
fig.add_vline(x=v["time_of_alert"], line_dash="dash", line_color="orange", annotation_text="alert")
fig.add_vline(x=v["time_of_event"], line_dash="dash", line_color="red", annotation_text="event")
fig.update_layout(height=300, showlegend=False)
fig.show()

<div dir="rtl">

**نتیجه.**
- ۵۲۹۵ پنجره‌ی خطر از ۷۲۶ ویدئو. ۲۴ ویدئوی تصادف پنجره‌ی خطر ندارند، چون در آن‌ها هشدار کمتر از ۰.۳ ثانیه قبل از حادثه آمده است.
- ۴۴۹۴ منفی سخت از ۷۴۹ ویدئوی تصادف و ۸۹۹۳ پنجره از ۷۵۰ ویدئوی عادی.
- سهم پنجره‌های خطر ۲۸٪ است (۵۲۹۵ از ۱۸٬۷۸۲).

</div>

## Evaluation Windows

<div dir="rtl">

قاعده همان قاعده‌ی آزمون رسمی است. برای هر Δ (۰.۵، ۱ و ۱.۵ ثانیه):
- **تصادف:** پنجره‌ای که Δ ثانیه قبل از حادثه تمام می‌شود، اگر هشدار انسانی زودتر از آن رسیده باشد.
- **رانندگی عادی:** پنجره‌ای که Δ ثانیه قبل از زمان حادثه‌ی ساختگی تمام می‌شود. اگر در ویدئو جا نشود، آن ویدئو برای آن Δ کنار می‌رود و به آخر ویدئو منتقل نمی‌شود، تا جای پنجره‌ها در دو دسته یکی بماند.

تعداد دو دسته برابر است، پس AP تصادفی ۰.۵ است. هر پنجره ۳ ثانیه فریم پشت سر دارد تا طول‌های مختلف پنجره بعداً روی پنجره‌های یکسان سنجیده شوند.

</div>

In [9]:
short = train.loc[normal, "last_time"] < 20
print("Normal videos shorter than 20 s:", short.sum(), "of", normal.sum())
rows = []
for delta in DELTAS:
    positives = train[(train["label"] == 1) & (train["gap"] > delta)]
    negatives = train[normal & (train["fake_event"] - delta <= train["last_time"])]
    n = min(len(positives), len(negatives))
    print(f"delta {delta}: collisions {len(positives)} | normal videos that fit {len(negatives)} | "
          f"kept {n} of each")
    for part in [positives.sample(n, random_state=SEED), negatives.sample(n, random_state=SEED)]:
        rows += [(v.video_path, v.event - delta, v.label, delta) for v in part.itertuples()]
eval_windows = pd.DataFrame(rows, columns=["video_path", "t_end", "label", "delta"])
eval_windows["end"] = end_frame(eval_windows, LONGEST_WINDOW)
print("Windows with less than 3 s of frames:", (eval_windows["end"] < 0).sum())
eval_windows = eval_windows[eval_windows["end"] >= 0].reset_index(drop=True)

print("\nEvaluation windows (rows: seconds before the event):")
print(pd.crosstab(eval_windows["delta"], eval_windows["label"]))
print("\nWindow end time in the video (s):")
print(eval_windows.groupby(["delta", "label"])["t_end"].describe()[["25%", "50%", "75%"]].round(2))

Normal videos shorter than 20 s: 105 of 750
delta 0.5: collisions 700 | normal videos that fit 661 | kept 661 of each
delta 1.0: collisions 556 | normal videos that fit 669 | kept 556 of each
delta 1.5: collisions 347 | normal videos that fit 686 | kept 347 of each
Windows with less than 3 s of frames: 6

Evaluation windows (rows: seconds before the event):
label    0    1
delta          
0.5    659  660
1.0    554  555
1.5    347  347

Window end time in the video (s):
               25%    50%    75%
delta label                     
0.5   0      18.52  19.30  19.83
      1      18.67  19.33  19.83
1.0   0      18.02  18.80  19.38
      1      18.17  18.85  19.36
1.5   0      17.50  18.20  18.81
      1      17.73  18.40  18.92


<div dir="rtl">

**نتیجه.**
- ۱۰۵ ویدئو از ۷۵۰ ویدئوی عادی کوتاه‌تر از ۲۰ ثانیه‌اند، در حالی که پنجره‌های ارزیابی تصادف به‌طور میانه حدود ثانیه‌ی ۱۹ تمام می‌شوند. برای همین در Δ برابر ۰.۵ فقط ۶۶۱ ویدئوی عادی جا شدند، کمتر از ۷۰۰ ویدئوی تصادف، و از هر دسته ۶۶۱ تا برداشته شد. در ۱ و ۱.۵ ثانیه ویدئوی عادی کافی بود.
- ۶ پنجره کمتر از ۳ ثانیه فریم داشتند و کنار رفتند. پنجره‌های نهایی: ۶۶۰ و ۶۵۹ در ۰.۵ ثانیه، ۵۵۵ و ۵۵۴ در ۱ ثانیه، ۳۴۷ و ۳۴۷ در ۱.۵ ثانیه (تصادف و عادی).
- جای پنجره‌ها در ویدئو در دو دسته تقریباً یکی است. میانه‌ی زمان پایان: ۱۹.۳۳ در برابر ۱۹.۳۰ ثانیه در Δ برابر ۰.۵، ۱۸.۸۵ در برابر ۱۸.۸۰ در ۱ ثانیه و ۱۸.۴۰ در برابر ۱۸.۲۰ در ۱.۵ ثانیه. بیشترین اختلاف چارک‌ها ۰.۲۳ ثانیه است.

</div>

## Folds

<div dir="rtl">

ویدئوها، نه پنجره‌ها، به پنج دسته تقسیم می‌شوند. پس همه‌ی پنجره‌های یک ویدئو یا در آموزش‌اند یا در ارزیابی. ویدئوهایی که در بررسی بالا صحنه‌ی یکسان داشتند در یک گروه قرار می‌گیرند و با هم به یک دسته می‌روند. سهم ویدئوهای تصادف در همه‌ی دسته‌ها یکی نگه داشته می‌شود.

</div>

In [10]:
group = dict(zip(train["video_path"], range(len(train))))
for a, b in zip(duplicates["video_path"], duplicates["nearest"]):
    old, new = group[b], group[a]
    group = {p: new if g == old else g for p, g in group.items()}
train["group"] = train["video_path"].map(group)
print("Train videos in a duplicate group:", train["group"].duplicated(keep=False).sum(),
      "| largest group:", train["group"].value_counts().max())

folds = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
for k, (_, held_out) in enumerate(folds.split(train, train["label"], groups=train["group"])):
    train.loc[held_out, "fold"] = k
train["fold"] = train["fold"].astype(int)
fold_of = train.set_index("video_path")["fold"]
windows["fold"] = windows["video_path"].map(fold_of)
eval_windows["fold"] = eval_windows["video_path"].map(fold_of)
print("Duplicate groups split across folds:", (train.groupby("group")["fold"].nunique() > 1).sum())

print("\nVideos per fold (columns: label):")
print(pd.crosstab(train["fold"], train["label"], margins=True))
print("\nEvaluation windows per fold (columns: label):")
print(pd.crosstab(eval_windows["fold"], eval_windows["label"]))

Train videos in a duplicate group: 2 | largest group: 2
Duplicate groups split across folds: 0

Videos per fold (columns: label):
label    0    1   All
fold                 
0      150  149   299
1      150  150   300
2      150  151   301
3      150  150   300
4      150  150   300
All    750  750  1500

Evaluation windows per fold (columns: label):
label    0    1
fold           
0      305  314
1      319  317
2      310  306
3      299  305
4      327  320


<div dir="rtl">

**نتیجه.**
- تنها گروه تکراری (`00788` و `00789`) کامل در یک دسته است.
- هر دسته ۲۹۹ تا ۳۰۱ ویدئو دارد که ۱۴۹ تا ۱۵۱ تای آن‌ها تصادف‌اند. هر دسته بین ۲۹۹ و ۳۲۷ پنجره‌ی ارزیابی از هر برچسب دارد.

</div>

## Models

<div dir="rtl">

پنج مدل مقایسه می‌شوند:
- **سه LightGBM** روی خلاصه‌ی پنجره: فریم آخر (بدون اطلاعات زمانی)، میانگین ۲۰ فریم، و میانگین به‌اضافه‌ی دو ویژگی حرکت (تغییر متوسط بین فریم‌های پشت‌سرهم و تفاضل فریم آخر و اول).
- **GRU و 1D CNN** روی خود دنباله‌ی ۲۰ فریم. GRU فریم‌ها را به ترتیب می‌خواند و CNN الگوهای کوتاه چندفریمی را پیدا می‌کند.

ترکیب بهترین مدل درختی و بهترین شبکه هم سنجیده می‌شود. یک LightGBM هم فقط نور، آب‌وهوا و صحنه را می‌بیند (با ۱۰۰ درخت ثابت)، تا معلوم شود بدون دیدن تصویر چه امتیازی به دست می‌آید.

در هر دسته، PCA بعد فریم‌ها را از ۱۹۲۰ به ۲۵۶ می‌رساند و ۱۵٪ از ویدئوهای آموزشی تعداد درخت‌ها یا epochها را تعیین می‌کنند. هر دو فقط از ویدئوهای آموزشیِ همان دسته یاد می‌گیرند.

</div>

In [11]:
PCA_DIM = 256
INNER_SHARE = 0.15
NET_LR = 3e-4
CACHE_DIR = DRIVE_DIR / "cache" / "v2"   # new version after changing windows, folds or models
(CACHE_DIR / "models").mkdir(parents=True, exist_ok=True)

MODELS = {"last": "LightGBM, last frame", "mean": "LightGBM, window mean",
          "motion": "LightGBM, mean + motion", "gru": "GRU", "cnn": "1D CNN",
          "meta": "LightGBM, metadata only"}
TREES = ["last", "mean", "motion"]
NETS = ["gru", "cnn"]

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", torch.cuda.get_device_name(0) if device == "cuda" else "cpu")

start_of = videos.set_index("video_path")["start"]
meta = pd.get_dummies(train[["light_conditions", "weather", "scene"]]).astype(np.float32)
META = dict(zip(train["video_path"], meta.to_numpy()))

Device: cpu


In [12]:
def offsets(length):
    return np.arange(-length + 1, 1)


def fit_pca(end):
    sample_rng = np.random.default_rng(SEED)
    frames = sample_rng.choice(end, 30_000) - sample_rng.integers(0, WINDOW_FRAMES, 30_000)
    return PCA(n_components=PCA_DIM, random_state=SEED).fit(FEATS[frames].astype(np.float32))


def project(pca):
    P = np.empty((len(FEATS), PCA_DIM), dtype=np.float32)
    for i in range(0, len(FEATS), 100_000):
        P[i:i + 100_000] = pca.transform(FEATS[i:i + 100_000].astype(np.float32))
    return P


def summarize(P, end, kind, length):
    parts = []
    for i in range(0, len(end), 5000):
        seq = P[end[i:i + 5000, None] + offsets(length)]
        if kind == "last":
            parts.append(seq[:, -1])
        elif kind == "mean":
            parts.append(seq.mean(axis=1))
        else:
            motion = np.abs(np.diff(seq, axis=1)).mean(axis=1)
            parts.append(np.hstack([seq.mean(axis=1), motion, seq[:, -1] - seq[:, 0]]))
    return np.concatenate(parts)


def fit_tree(X, y, X_val=None, y_val=None, n_trees=None):
    model = lgb.LGBMClassifier(n_estimators=n_trees or 2000, learning_rate=0.03, num_leaves=31,
                               subsample=0.8, subsample_freq=1, colsample_bytree=0.5,
                               metric="average_precision", random_state=SEED, verbose=-1)
    if n_trees:
        return model.fit(X, y)
    stop = lgb.early_stopping(100, verbose=False)
    return model.fit(X, y, eval_set=[(X_val, y_val)], callbacks=[stop])

In [13]:
class GRUNet(nn.Module):
    def __init__(self, hidden=128):
        super().__init__()
        self.inp = nn.Sequential(nn.LayerNorm(PCA_DIM), nn.Linear(PCA_DIM, 256), nn.GELU(),
                                 nn.Dropout(0.3))
        self.gru = nn.GRU(256, hidden, batch_first=True)
        self.out = nn.Sequential(nn.Dropout(0.3), nn.Linear(hidden, 1))

    def forward(self, x):
        h, _ = self.gru(self.inp(x))
        return self.out(h[:, -1]).squeeze(-1)


class ConvNet(nn.Module):
    def __init__(self, d=256):
        super().__init__()
        self.inp = nn.Sequential(nn.LayerNorm(PCA_DIM), nn.Linear(PCA_DIM, d), nn.GELU(),
                                 nn.Dropout(0.3))
        self.conv = nn.Sequential(nn.Conv1d(d, d, 3, padding=1), nn.GELU(),
                                  nn.Conv1d(d, d, 3, padding=2, dilation=2), nn.GELU())
        self.out = nn.Sequential(nn.Dropout(0.3), nn.Linear(2 * d, 1))

    def forward(self, x):
        h = self.conv(self.inp(x).transpose(1, 2))
        return self.out(torch.cat([h.mean(dim=2), h[:, :, -1]], dim=1)).squeeze(-1)


@torch.inference_mode()
def predict_net(model, P, end, length):
    model.to(device).eval()
    scores = []
    for i in range(0, len(end), 2048):
        x = torch.from_numpy(P[end[i:i + 2048, None] + offsets(length)]).to(device)
        scores.append(torch.sigmoid(model(x)).cpu().numpy())
    return np.concatenate(scores)


def fit_net(net_class, P, end, y, val_end, val_y, length, epochs=None):
    torch.manual_seed(SEED)
    order_rng = np.random.default_rng(SEED)
    model = net_class().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=NET_LR, weight_decay=1e-2)
    best_ap, best_state, best_epoch = -1, None, epochs
    for epoch in range(epochs or 30):
        model.train()
        order = order_rng.permutation(len(end))
        for i in range(0, len(end), 256):
            batch = order[i:i + 256]
            x = torch.from_numpy(P[end[batch, None] + offsets(length)]).to(device)
            target = torch.from_numpy(y[batch].astype(np.float32)).to(device)
            loss = F.binary_cross_entropy_with_logits(model(x), target)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        if epochs is None:
            ap = average_precision_score(val_y, predict_net(model, P, val_end, length))
            if ap > best_ap:
                best_ap, best_epoch = ap, epoch + 1
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
            elif epoch + 1 - best_epoch >= 5:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    model.best_epoch = best_epoch
    return model.cpu()

In [14]:
def fit_model(key, P, rows, val_rows, length):
    y, val_y = rows["label"].to_numpy(), val_rows["label"].to_numpy()
    if key == "meta":
        return fit_tree(meta_features(rows), y, n_trees=100), 100
    if key in TREES:
        X = summarize(P, rows["end"].to_numpy(), key, length)
        X_val = summarize(P, val_rows["end"].to_numpy(), key, length)
        model = fit_tree(X, y, X_val, val_y)
        return model, model.best_iteration_
    net_class = GRUNet if key == "gru" else ConvNet
    end, val_end = rows["end"].to_numpy(), val_rows["end"].to_numpy()
    model = fit_net(net_class, P, end, y, val_end, val_y, length)
    return model, model.best_epoch


def predict(key, model, P, rows, length):
    if key == "meta":
        return model.predict_proba(meta_features(rows))[:, 1]
    if key in TREES:
        return model.predict_proba(summarize(P, rows["end"].to_numpy(), key, length))[:, 1]
    return predict_net(model, P, rows["end"].to_numpy(), length)


def meta_features(rows):
    return np.stack([META[p] for p in rows["video_path"]])


def save_model(key, model, path):
    if key in NETS:
        torch.save(model.state_dict(), path.with_suffix(".pt"))
    else:
        joblib.dump(model, path.with_suffix(".joblib"))


def load_model(key, path):
    if key in NETS:
        model = GRUNet() if key == "gru" else ConvNet()
        model.load_state_dict(torch.load(path.with_suffix(".pt"), map_location="cpu"))
        return model
    return joblib.load(path.with_suffix(".joblib"))


def frames_before(rows):
    return rows["end"] - rows["video_path"].map(start_of) + 1

In [15]:
def run_fold(k, keys, length=WINDOW_FRAMES):
    pending = [key for key in keys if not (CACHE_DIR / f"{key}_{length}f_fold{k}.csv").exists()]
    started = time.time()
    fold_videos = train[train["fold"] != k]
    groups = fold_videos.drop_duplicates("group")
    inner_groups = groups.groupby("label").sample(frac=INNER_SHARE, random_state=SEED + k)["group"]
    inner = set(fold_videos.loc[fold_videos["group"].isin(inner_groups), "video_path"])

    fit_rows = windows[(windows["fold"] != k) & ~windows["video_path"].isin(inner)]
    fit_rows = fit_rows[frames_before(fit_rows) >= length]
    val_rows = eval_windows[eval_windows["video_path"].isin(inner)]
    same_video = windows[(windows["fold"] == k) & windows["kind"].isin(["risk", "hard negative"])]
    same_video = same_video[frames_before(same_video) >= LONGEST_WINDOW]
    held = pd.concat([eval_windows[eval_windows["fold"] == k].assign(set="evaluation"),
                      same_video.assign(set="same video")], ignore_index=True)

    pca_path = CACHE_DIR / f"pca_fold{k}.joblib"
    if not pca_path.exists():
        joblib.dump(fit_pca(fit_rows["end"].to_numpy()), pca_path)
    pca = joblib.load(pca_path)
    print(f"Fold {k + 1} | windows: fit {len(fit_rows)}, early stopping {len(val_rows)}, "
          f"held out {len(held)} | PCA variance kept {pca.explained_variance_ratio_.sum():.1%}")
    if not pending:
        return
    P = project(pca)
    for key in pending:
        model, size = fit_model(key, P, fit_rows, val_rows, length)
        save_model(key, model, CACHE_DIR / "models" / f"{key}_{length}f_fold{k}")
        scores = held.assign(score=predict(key, model, P, held, length), size=size)
        scores.to_csv(CACHE_DIR / f"{key}_{length}f_fold{k}.csv", index=False)
        print(f"  {MODELS[key]} | trees/epochs: {size} | {time.time() - started:.0f}s")


started = time.time()
for k in range(N_FOLDS):
    run_fold(k, list(MODELS))
print(f"Cross-validation: {(time.time() - started) / 60:.1f} min")

Fold 1 | windows: fit 12739, early stopping 382, held out 2605 | PCA variance kept 91.6%
Fold 2 | windows: fit 12810, early stopping 387, held out 2551 | PCA variance kept 91.6%
Fold 3 | windows: fit 12736, early stopping 383, held out 2525 | PCA variance kept 91.6%
Fold 4 | windows: fit 12836, early stopping 375, held out 2527 | PCA variance kept 91.6%
Fold 5 | windows: fit 12717, early stopping 369, held out 2619 | PCA variance kept 91.5%
Cross-validation: 0.0 min


## Cross-Validation Results

<div dir="rtl">

هر مدل پنج بار آموزش می‌بیند و هر بار فقط به ویدئوهای دسته‌ی کنار گذاشته امتیاز می‌دهد.
- **mAP:** معیار رسمی چالش. بازه‌ی ۹۵٪ با bootstrap روی ویدئوها ساخته می‌شود، نه روی پنجره‌ها، چون پنجره‌های یک ویدئو به هم وابسته‌اند.
- **AUC داخل یک ویدئو:** برای هر ویدئوی تصادف، پنجره‌های خطر با منفی‌های سختِ همان ویدئو مقایسه می‌شوند و از AUC ویدئوها میانگین گرفته می‌شود. صحنه، نور و دوربین در دو طرف یکی است، پس فرق بین ویدئوها روی این عدد اثری ندارد.

مدلی که بیشترین mAP را دارد انتخاب می‌شود. مدل متادیتا فقط برای مقایسه است. انتخاب بهترین از بین چند مدل روی همان داده کمی خوش‌بینانه است، پس عدد اصلی گزارش آزمون رسمی است.

</div>

In [16]:
def load_scores(keys, length=WINDOW_FRAMES):
    # out-of-fold scores of one model, or the average of several (ensemble)
    tables = []
    for key in keys:
        parts = [pd.read_csv(CACHE_DIR / f"{key}_{length}f_fold{k}.csv") for k in range(N_FOLDS)]
        tables.append(pd.concat(parts, ignore_index=True))
    scores = tables[0].copy()
    scores["score"] = np.mean([t["score"] for t in tables], axis=0)
    return scores


def mean_ap(table):
    aps = [average_precision_score(g["label"], g["score"]) for _, g in table.groupby("delta")]
    return np.mean(aps)


def bootstrap_ci(table, n=1000):
    boot_rng = np.random.default_rng(SEED)
    rows_of = table.groupby("video_path").indices
    paths = list(rows_of)
    values = []
    for _ in range(n):
        sample = boot_rng.choice(len(paths), len(paths))
        values.append(mean_ap(table.iloc[np.concatenate([rows_of[paths[i]] for i in sample])]))
    return np.quantile(values, [0.025, 0.975])


def same_video_auc(same):
    # AUC inside each collision video, averaged over videos
    aucs = [roc_auc_score(g["label"], g["score"]) for _, g in same.groupby("video_path")
            if g["label"].nunique() == 2]
    return np.mean(aucs)


def summary_row(name, scores):
    ev = scores[scores["set"] == "evaluation"]
    same = scores[scores["set"] == "same video"]
    low, high = bootstrap_ci(ev)
    row = {"model": name, "mAP": mean_ap(ev), "CI low": low, "CI high": high}
    for d, g in ev.groupby("delta"):
        row[f"AP@{d}s"] = average_precision_score(g["label"], g["score"])
    row["AUC"] = roc_auc_score(ev["label"], ev["score"])
    row["same-video AUC"] = same_video_auc(same)
    return row


scores = {key: load_scores([key]) for key in MODELS}
sizes = {key: int(scores[key].groupby("fold")["size"].first().median()) for key in MODELS}
print("Trees or epochs chosen in each fold:")
print(pd.DataFrame({MODELS[key]: scores[key].groupby("fold")["size"].first() for key in MODELS}))
image_models = TREES + NETS
best_tree = max(TREES, key=lambda key: mean_ap(scores[key][scores[key]["set"] == "evaluation"]))
best_net = max(NETS, key=lambda key: mean_ap(scores[key][scores[key]["set"] == "evaluation"]))
scores["ensemble"] = load_scores([best_tree, best_net])
MODELS["ensemble"] = f"{MODELS[best_tree]} + {MODELS[best_net]}"

summary = pd.DataFrame([summary_row(MODELS[key], scores[key]) for key in scores])
summary["trees/epochs"] = [sizes.get(key) for key in scores]
summary = summary.sort_values("mAP", ascending=False).reset_index(drop=True)
summary.to_csv(RESULT_DIR / "cv_comparison.csv", index=False)
same = scores["last"][scores["last"]["set"] == "same video"]
print("Collision videos in the same-video AUC:",
      (same.groupby("video_path")["label"].nunique() == 2).sum())
summary.round(3)

Trees or epochs chosen in each fold:
      LightGBM, last frame  LightGBM, window mean  LightGBM, mean + motion  \
fold                                                                         
0                       50                     86                      180   
1                      317                     44                       59   
2                       68                    159                       88   
3                      171                     23                       25   
4                       69                     72                       63   

      GRU  1D CNN  LightGBM, metadata only  
fold                                        
0       1       1                      100  
1       1       1                      100  
2       1       1                      100  
3      15      10                      100  
4       3       2                      100  
Collision videos in the same-video AUC: 724


,model,mAP,CI low,CI high,AP@0.5s,AP@1.0s,AP@1.5s,AUC,same-video AUC,trees/epochs
0,"LightGBM, last frame + 1D CNN",0.731,0.692,0.769,0.726,0.733,0.734,0.753,0.682,NaN
1,"LightGBM, last frame",0.724,0.687,0.764,0.726,0.733,0.714,0.741,0.653,69.0
2,1D CNN,0.706,0.666,0.746,0.703,0.704,0.711,0.724,0.672,1.0
3,GRU,0.703,0.661,0.742,0.695,0.702,0.710,0.704,0.678,1.0
4,"LightGBM, mean + motion",0.701,0.663,0.742,0.729,0.699,0.675,0.718,0.686,63.0
5,"LightGBM, window mean",0.669,0.625,0.712,0.659,0.674,0.674,0.683,0.565,72.0
6,"LightGBM, metadata only",0.576,0.535,0.619,0.579,0.578,0.569,0.604,0.500,100.0


In [17]:
chart = summary.sort_values("mAP")
fig = go.Figure(go.Bar(x=chart["mAP"], y=chart["model"], orientation="h",
                       error_x=dict(type="data", array=chart["CI high"] - chart["mAP"],
                                    arrayminus=chart["mAP"] - chart["CI low"])))
fig.add_vline(x=0.5, line_dash="dot", annotation_text="random")
fig.update_layout(height=400, title="Cross-validation mAP with 95% bootstrap interval",
                  xaxis=dict(title="mAP", range=[0.4, 1]))
fig.show()

fig = px.scatter(summary, x="same-video AUC", y="mAP", color="model",
                 title="Overall score against the same-video test")
fig.add_vline(x=0.5, line_dash="dot")
fig.update_traces(marker_size=12)
fig.update_layout(height=450, xaxis_range=[0.4, 1], yaxis_range=[0.4, 1])
fig.show()

candidates = summary[summary["model"] != MODELS["meta"]]
FINAL = next(key for key in scores if MODELS[key] == candidates.iloc[0]["model"])
FINAL_PARTS = [best_tree, best_net] if FINAL == "ensemble" else [FINAL]
print("Selected model:", MODELS[FINAL])

Selected model: LightGBM, last frame + 1D CNN


<div dir="rtl">

**نتیجه.**
- بهترین مدل ترکیب «LightGBM، فریم آخر» و 1D CNN است: mAP برابر ۰.۷۳۱ (بازه‌ی ۹۵٪: ۰.۶۹۲ تا ۰.۷۶۹)، AUC برابر ۰.۷۵۳، و AP در سه گروه ۰.۷۲۶، ۰.۷۳۳ و ۰.۷۳۴.
- مدل متادیتا بدون دیدن تصویر به ۰.۵۷۶ می‌رسد، پس فرق بین ویدئوها به‌تنهایی mAP را از ۰.۵ به ۰.۵۷۶ می‌برد. همه‌ی مدل‌های تصویری (۰.۶۶۹ تا ۰.۷۳۱) بالاتر از آن‌اند و بازه‌هایشان با بازه‌ی آن (تا ۰.۶۱۹) هم‌پوشانی ندارد.
- فاصله‌ی پنج مدل اول (۰.۷۰۱ تا ۰.۷۳۱) از پهنای هر بازه (حدود ۰.۰۸) خیلی کمتر است، و CNN فقط ۰.۰۰۳ از GRU بهتر است.
- AUC داخل یک ویدئو برای مدل متادیتا دقیقاً ۰.۵۰۰ است، چون به همه‌ی پنجره‌های یک ویدئو یک امتیاز می‌دهد. مدل‌های تصویری بین ۰.۵۶۵ و ۰.۶۸۶ هستند، پس بخشی از خودِ خطر را می‌بینند.
- فریم آخر (۰.۷۲۴) از میانگین ۲۰ فریم (۰.۶۶۹) بهتر است، چون میانگین وضع لحظه‌ی آخر را محو می‌کند. حرکت، AUC داخل یک ویدئوی میانگین را از ۰.۵۶۵ به ۰.۶۸۶ می‌رساند، که بیشترین مقدار بین مدل‌هاست.
- بهترین epoch در سه دسته از پنج دسته اولی بود (GRU: ۱، ۱، ۱، ۱۵، ۳؛ CNN: ۱، ۱، ۱، ۱۰، ۲). پس مدل نهایی شبکه را ۱ epoch آموزش می‌دهد.
- داده‌ی توقف زودهنگام ۳۶۹ تا ۳۸۷ پنجره است و تعداد درخت‌ها بین دسته‌ها خیلی فرق می‌کند (فریم آخر: ۵۰ تا ۳۱۷). PCA ۹۱.۵ تا ۹۱.۶٪ واریانس را نگه داشت.

</div>

## Window Length

<div dir="rtl">

طول پنجره تا اینجا ۲ ثانیه بوده است. همین مدل انتخاب‌شده با پنجره‌ی ۱ و ۳ ثانیه هم آموزش می‌بیند، روی همان دسته‌ها و همان پنجره‌های ارزیابی. قاعده از قبل تعیین شده تا انتخاب شانسی نباشد: طول دیگری فقط وقتی جای ۲ ثانیه را می‌گیرد که mAP آن بیشتر باشد و در دست‌کم ۴ دسته از ۵ دسته هم بهتر باشد.

</div>

In [18]:
LENGTHS = [10, 20, 30]
for length in LENGTHS:
    for k in range(N_FOLDS):
        run_fold(k, FINAL_PARTS, length)

tables = {length: load_scores(FINAL_PARTS, length) for length in LENGTHS}
per_fold = {length: t[t["set"] == "evaluation"].groupby("fold").apply(mean_ap, include_groups=False)
            for length, t in tables.items()}
rows = []
for length, table in tables.items():
    row = summary_row(f"{length / 10:.0f} s", table)
    row["folds better than 2 s"] = (per_fold[length] > per_fold[WINDOW_FRAMES]).sum()
    rows.append(row)
lengths = pd.DataFrame(rows, index=LENGTHS).rename(columns={"model": "window"})
lengths.to_csv(RESULT_DIR / "window_comparison.csv", index=False)

base = lengths.loc[WINDOW_FRAMES, "mAP"]
better = lengths[(lengths["mAP"] > base) & (lengths["folds better than 2 s"] >= 4)]
FINAL_LENGTH = better["mAP"].idxmax() if len(better) else WINDOW_FRAMES
print("Model:", MODELS[FINAL], "| window used from here on:", FINAL_LENGTH / 10, "s")
print("\nmAP of each fold:")
print(pd.DataFrame({f"{n / 10:.0f} s": per_fold[n] for n in LENGTHS}).round(3))
lengths.round(3)

Fold 1 | windows: fit 12739, early stopping 382, held out 2605 | PCA variance kept 91.6%
Fold 2 | windows: fit 12810, early stopping 387, held out 2551 | PCA variance kept 91.6%
Fold 3 | windows: fit 12736, early stopping 383, held out 2525 | PCA variance kept 91.6%
Fold 4 | windows: fit 12836, early stopping 375, held out 2527 | PCA variance kept 91.6%
Fold 5 | windows: fit 12717, early stopping 369, held out 2619 | PCA variance kept 91.5%
Fold 1 | windows: fit 12739, early stopping 382, held out 2605 | PCA variance kept 91.6%
Fold 2 | windows: fit 12810, early stopping 387, held out 2551 | PCA variance kept 91.6%
Fold 3 | windows: fit 12736, early stopping 383, held out 2525 | PCA variance kept 91.6%
Fold 4 | windows: fit 12836, early stopping 375, held out 2527 | PCA variance kept 91.6%
Fold 5 | windows: fit 12717, early stopping 369, held out 2619 | PCA variance kept 91.5%
Fold 1 | windows: fit 12581, early stopping 382, held out 2605 | PCA variance kept 91.6%
Fold 2 | windows: fit

,window,mAP,CI low,CI high,AP@0.5s,AP@1.0s,AP@1.5s,AUC,same-video AUC,folds better than 2 s
10,1 s,0.742,0.705,0.779,0.739,0.744,0.742,0.765,0.684,3
20,2 s,0.731,0.692,0.769,0.726,0.733,0.734,0.753,0.682,0
30,3 s,0.727,0.689,0.767,0.738,0.730,0.714,0.749,0.681,4


<div dir="rtl">

**نتیجه.** طبق قاعده، پنجره‌ی ۲ ثانیه‌ای می‌ماند (mAP برابر ۰.۷۳۱).
- پنجره‌ی ۱ ثانیه‌ای بیشترین mAP را دارد (۰.۷۴۲)، ولی فقط در ۳ دسته از ۵ دسته بهتر بود.
- پنجره‌ی ۳ ثانیه‌ای در ۴ دسته بهتر بود، ولی mAP کلی‌اش کمتر است (۰.۷۲۷).
- بازه‌های هر سه تقریباً یکی‌اند و AUC داخل یک ویدئو بین ۰.۶۸۱ و ۰.۶۸۴ است. پس طول پنجره در این بازه اثر روشنی ندارد.
- فریم آخرِ پنجره‌ی ۱ و ۲ ثانیه‌ای یکی است و پنجره‌های آموزشی هم یکی‌اند (تعداد fit در هر دسته برابر است). پس بخش LightGBM در این دو طول یک مدل است و فرقشان فقط از CNN می‌آید.

</div>

## Out-of-Fold Risk Curves

<div dir="rtl">

برای تحلیل، هر ویدئوی train یک منحنی خطر می‌گیرد: امتیاز مدل انتخاب‌شده هر ۰.۲ ثانیه، از اولین لحظه‌ای که ۲ ثانیه فریم پشت سرش هست تا آخر ویدئو. هر ویدئو فقط با PCA و مدل‌های دسته‌ای امتیاز می‌گیرد که آن را ندیده‌اند. پس منحنی‌ها out-of-fold هستند و برای ویدئوهای نمونه‌ی داشبورد هم قابل استفاده‌اند. پنجره‌های بخش بعد (زمان واکنش) هم در همین حلقه امتیاز می‌گیرند.

برای اطمینان از اینکه مدل‌های ذخیره‌شده همان مدل‌های جدول مقایسه‌اند، امتیاز پنجره‌های ارزیابی دوباره حساب و با امتیاز ذخیره‌شده مقایسه می‌شود.

</div>

In [19]:
from plotly.subplots import make_subplots
from statsmodels.stats.proportion import proportion_confint

final_scores = load_scores(FINAL_PARTS, FINAL_LENGTH)


def final_score(models, P, rows):
    return np.mean([predict(key, models[key], P, rows, FINAL_LENGTH) for key in FINAL_PARTS],
                   axis=0)


parts = []
for v in train.itertuples():
    s, n = VIDEO_FRAMES[v.video_path]
    ends = np.arange(s + FINAL_LENGTH - 1, s + n, 2)
    parts.append(pd.DataFrame({"video_path": v.video_path, "fold": v.fold, "label": v.label,
                               "t_end": TIMES[ends], "end": ends}))
curves = pd.concat(parts, ignore_index=True)

REACTION_DELTAS = np.round(np.arange(0.2, 3.01, 0.1), 1)
rows = []
for delta in REACTION_DELTAS:
    positives = train[(train["label"] == 1) & (train["gap"] > delta)]
    negatives = train[normal & (train["fake_event"] - delta <= train["last_time"])]
    n = min(len(positives), len(negatives))
    for part in [positives.sample(n, random_state=SEED), negatives.sample(n, random_state=SEED)]:
        rows += [(v.video_path, v.fold, v.event - delta, v.label, delta) for v in part.itertuples()]
reaction = pd.DataFrame(rows, columns=["video_path", "fold", "t_end", "label", "delta"])
reaction["end"] = end_frame(reaction, LONGEST_WINDOW)
reaction = reaction[reaction["end"] >= 0].reset_index(drop=True)

curve_scores, reaction_scores, largest_diff = [], [], 0.0
for k in range(N_FOLDS):
    P = project(joblib.load(CACHE_DIR / f"pca_fold{k}.joblib"))
    models = {key: load_model(key, CACHE_DIR / "models" / f"{key}_{FINAL_LENGTH}f_fold{k}")
              for key in FINAL_PARTS}
    saved = final_scores[final_scores["fold"] == k]
    largest_diff = max(largest_diff, np.abs(final_score(models, P, saved) - saved["score"]).max())
    for table, out in [(curves, curve_scores), (reaction, reaction_scores)]:
        part = table[table["fold"] == k]
        out.append(pd.Series(final_score(models, P, part), index=part.index))
    del P
curves["score"] = pd.concat(curve_scores)
reaction["score"] = pd.concat(reaction_scores)

curves.drop(columns="end").round({"t_end": 3, "score": 5}).to_csv(
    RESULT_DIR / "oof_curves.csv.gz", index=False)
print(f"Curve points: {len(curves)} from {curves['video_path'].nunique()} videos")
print(f"Largest difference between recomputed and saved cross-validation scores: "
      f"{largest_diff:.1e}")

Curve points: 204752 from 1500 videos
Largest difference between recomputed and saved cross-validation scores: 1.5e-08


<div dir="rtl">

**نتیجه.**
- ۲۰۴٬۷۵۲ نقطه‌ی منحنی از هر ۱۵۰۰ ویدئوی train ساخته شد.
- بیشترین اختلاف امتیاز دوباره حساب‌شده با امتیاز ذخیره‌شده‌ی جدول مقایسه 1.5e-08 است. پس مدل‌ها و PCA بارشده همان‌هایی‌اند که جدول مقایسه را ساختند.

</div>

## Reaction Time

<div dir="rtl">

جدول مقایسه فقط سه لحظه را می‌سنجد: ۰.۵، ۱ و ۱.۵ ثانیه قبل از حادثه. اینجا همان قاعده برای δ از ۰.۲ تا ۳ ثانیه، هر ۰.۱ ثانیه، تکرار می‌شود. پنجره‌ی تصادف δ ثانیه قبل از حادثه تمام می‌شود، فقط اگر هشدار انسانی زودتر از آن آمده باشد. پنجره‌ی عادی δ ثانیه قبل از زمان حادثه‌ی ساختگی تمام می‌شود و تعداد دو دسته برابر است.

در δ بزرگ فقط ویدئوهایی می‌مانند که خطر در آن‌ها زود دیده می‌شد، پس نمونه کوچک‌تر و خاص‌تر است. بازه‌ی ۹۵٪ با bootstrap روی ویدئوهاست.

</div>

In [20]:
boot_rng = np.random.default_rng(SEED)
rows = []
for delta, g in reaction.groupby("delta"):
    y, s = g["label"].to_numpy(), g["score"].to_numpy()
    boot = []
    for _ in range(1000):
        i = boot_rng.choice(len(g), len(g))
        boot.append(average_precision_score(y[i], s[i]))
    rows.append({"delta": delta, "collisions": int(y.sum()), "AP": average_precision_score(y, s),
                 "CI low": np.quantile(boot, 0.025), "CI high": np.quantile(boot, 0.975)})
reaction_curve = pd.DataFrame(rows)
reaction_curve.to_csv(RESULT_DIR / "reaction_curve.csv", index=False)

evaluation = final_scores[final_scores["set"] == "evaluation"]
for d in DELTAS:
    in_table = evaluation[evaluation["delta"] == d]
    here = reaction_curve.loc[reaction_curve["delta"] == d, "AP"].iloc[0]
    print(f"delta {d}: AP here {here:.4f} | in the comparison table "
          f"{average_precision_score(in_table['label'], in_table['score']):.4f}")

x = reaction_curve["delta"]
fig = go.Figure([
    go.Scatter(x=x, y=reaction_curve["CI high"], line_width=0, showlegend=False, hoverinfo="skip"),
    go.Scatter(x=x, y=reaction_curve["CI low"], line_width=0, fill="tonexty", name="95% interval",
               fillcolor="rgba(70, 130, 180, 0.2)", hoverinfo="skip"),
    go.Scatter(x=x, y=reaction_curve["AP"], mode="lines+markers", name="AP", line_color="steelblue",
               customdata=reaction_curve["collisions"],
               hovertemplate="δ %{x} s<br>AP %{y:.3f}<br>collisions %{customdata}")])
fig.add_hline(y=0.5, line_dash="dot", annotation_text="random")
fig.update_layout(height=400, title="AP by time before the event (cross-validation)",
                  xaxis_title="seconds before the event (δ)", yaxis_title="AP",
                  yaxis_range=[0.3, 1])
fig.show()
reaction_curve.round(3)

delta 0.5: AP here 0.7259 | in the comparison table 0.7259
delta 1.0: AP here 0.7326 | in the comparison table 0.7326
delta 1.5: AP here 0.7335 | in the comparison table 0.7335


,delta,collisions,AP,CI low,CI high
0,0.2,657,0.739,0.701,0.776
1,0.3,657,0.741,0.704,0.778
2,0.4,658,0.742,0.709,0.777
3,0.5,660,0.726,0.690,0.763
4,0.6,662,0.730,0.692,0.768
5,0.7,662,0.730,0.692,0.768
6,0.8,643,0.729,0.692,0.765
7,0.9,605,0.729,0.690,0.767
8,1.0,555,0.733,0.692,0.774
9,1.1,517,0.728,0.685,0.770


<div dir="rtl">

**نتیجه.**
- AP در ۰.۵، ۱ و ۱.۵ ثانیه (۰.۷۲۵۹، ۰.۷۳۲۶ و ۰.۷۳۳۵) با جدول مقایسه یکی است.
- از ۰.۲ تا ۱.۵ ثانیه، AP بین ۰.۷۲۶ و ۰.۷۴۲ می‌ماند. AP در ۰.۲ ثانیه قبل از حادثه (۰.۷۳۹) از ۱.۵ ثانیه قبل (۰.۷۳۴) بیشتر نیست، پس با نزدیک شدن به حادثه زیاد نمی‌شود.
- از ۱.۵ تا ۳ ثانیه، تعداد تصادف‌ها از ۳۴۷ به ۶۲ می‌رسد و AP بین ۰.۶۸۰ و ۰.۷۴۹ نوسان می‌کند. بازه‌ها پهن می‌شوند (در ۳ ثانیه: ۰.۶۳۶ تا ۰.۸۷۲) و بازه‌ی همه‌ی δها با هم هم‌پوشانی دارد، پس روند روشنی دیده نمی‌شود.

</div>

## Alarm Threshold

<div dir="rtl">

mAP فقط ترتیب امتیازها را می‌سنجد، ولی سیستم هشدار به آستانه نیاز دارد. آستانه امتیازی است که ۸۰٪ پنجره‌های تصادف در ۱ ثانیه قبل از حادثه به آن برسند. هر دسته مدل‌های خودش را دارد و مقیاس امتیازشان می‌تواند فرق کند (مثلاً ۵۰ درخت در برابر ۳۱۷ درخت)، پس آستانه در هر دسته جدا، از امتیازهای out-of-fold همان دسته، تعیین می‌شود.

بعد هشدارهای اشتباه روی منحنی ویدئوهای عادی و بخش امن ویدئوهای تصادف (تا ۱ ثانیه قبل از `time_of_alert`) شمرده می‌شوند، هر ویدئو با آستانه‌ی دسته‌ی خودش. رسیدن امتیاز به آستانه یک هشدار است، ولی هشدار بعدی فقط وقتی شمرده می‌شود که دست‌کم ۳ ثانیه بعد باشد، چون هشدار واقعی هم چند ثانیه روی صفحه می‌ماند. نمودار همین حساب را در رانندگی عادی برای recallهای دیگر نشان می‌دهد.

</div>

In [21]:
at_1s = evaluation[evaluation["delta"] == 1.0]
risk_1s = at_1s[at_1s["label"] == 1]


def fold_thresholds(recall):
    # in each fold, the score reached by `recall` of its collision windows at 1 s
    return risk_1s.groupby("fold")["score"].quantile(1 - recall)


THRESHOLD = fold_thresholds(0.8)
THRESHOLD.to_csv(RESULT_DIR / "fold_thresholds.csv")
curves["above"] = curves["score"] >= curves["fold"].map(THRESHOLD)
above_1s = at_1s["score"] >= at_1s["fold"].map(THRESHOLD)
print("Threshold in each fold:", THRESHOLD.round(4).tolist())
print(f"Collisions above the threshold at 1 s: {above_1s[at_1s['label'] == 1].mean():.1%}")
print("\nWindows at 1 s above the threshold, by fold (columns: label):")
print(above_1s.groupby([at_1s["fold"], at_1s["label"]]).mean().unstack().round(3))
print()

pos_curves = curves[curves["label"] == 1].merge(
    train[["video_path", "time_of_alert", "time_of_event"]], on="video_path")
footage = {"normal driving": curves[curves["label"] == 0],
           "collision videos up to alert - 1 s":
               pos_curves[pos_curves["t_end"] <= pos_curves["time_of_alert"] - BUFFER]}
footage_arrays = {name: [(g["t_end"].to_numpy(), g["score"].to_numpy(), fold)
                         for (_, fold), g in table.groupby(["video_path", "fold"])]
                  for name, table in footage.items()}


def count_alarms(t, s, threshold, gap=3.0):
    # a new alarm needs at least `gap` seconds after the previous one
    count, last = 0, -np.inf
    for ti, si in zip(t, s):
        if si >= threshold and ti - last >= gap:
            count, last = count + 1, ti
    return count


def false_alarms(arrays, thresholds):
    count = sum(count_alarms(t, s, thresholds[fold]) for t, s, fold in arrays)
    hours = sum(t[-1] - t[0] + 0.2 for t, _, _ in arrays) / 3600
    return count, hours


rows = []
for name, table in footage.items():
    count, hours = false_alarms(footage_arrays[name], THRESHOLD)
    rows.append({"footage": name, "videos": table["video_path"].nunique(), "hours": hours,
                 "time above threshold": table["above"].mean(),
                 "false alarms": count, "per hour": count / hours})
alarms = pd.DataFrame(rows)
print(alarms.round(3).to_string(index=False))

rows = []
for recall in np.round(np.arange(0.3, 0.96, 0.05), 2):
    count, hours = false_alarms(footage_arrays["normal driving"], fold_thresholds(recall))
    rows.append({"recall at 1 s": recall, "false alarms per hour": count / hours})
tradeoff = pd.DataFrame(rows)
tradeoff.to_csv(RESULT_DIR / "alarm_tradeoff.csv", index=False)

fig = px.line(tradeoff, x="false alarms per hour", y="recall at 1 s", markers=True,
              title="Recall at 1 s against false alarms per hour of normal driving")
fig.add_scatter(x=[alarms.loc[0, "per hour"]], y=[0.8], mode="markers", marker_size=14,
                marker_symbol="star", name="chosen threshold")
fig.update_layout(height=400)
fig.show()

Threshold in each fold: [0.2307, 0.1989, 0.1964, 0.1108, 0.227]
Collisions above the threshold at 1 s: 79.8%

Windows at 1 s above the threshold, by fold (columns: label):
label      0      1
fold               
0      0.321  0.798
1      0.339  0.802
2      0.436  0.794
3      0.462  0.798
4      0.333  0.798

                           footage  videos  hours  time above threshold  false alarms   per hour
                    normal driving     750  7.352                 0.384          4522 615.054993
collision videos up to alert - 1 s     749  3.064                 0.648          2972 970.085999


<div dir="rtl">

**نتیجه.**
- آستانه‌ی دسته‌ها بین ۰.۱۱۱ و ۰.۲۳۱ است، یعنی مقیاس امتیازها بین دسته‌ها تا حدود دو برابر فرق دارد. recall در ۱ ثانیه در هر دسته بین ۷۹.۴ و ۸۰.۲٪ است (کل ۷۹.۸٪). با همین recall، سهم پنجره‌های عادیِ بالای آستانه در دسته‌ها از ۳۲.۱ تا ۴۶.۲٪ است.
- در ۷.۳۵ ساعت رانندگی عادی، امتیاز ۳۸.۴٪ زمان بالای آستانه است: ۴۵۲۲ هشدار اشتباه، یعنی ۶۱۵ بار در ساعت. در بخش امن ویدئوهای تصادف، که آن هم خطری ندارد، ۶۴.۸٪ زمان بالای آستانه است و ۹۷۰ بار در ساعت هشدار اشتباه داده می‌شود.
- در نمودار، حتی با recall برابر ۳۰٪ هشدار اشتباه بیشتر از ۱۰۰ بار در ساعت است. mAP فقط ترتیب امتیازها را در سه لحظه می‌سنجد، و mAP حدود ۰.۷۳ هنوز با یک سیستم هشدار قابل استفاده فاصله‌ی زیادی دارد.

</div>

## Example Curves

<div dir="rtl">

منحنی خطر هشت ویدئوی نمونه‌ی داشبورد: چهار رانندگی عادی و چهار تصادف. این ویدئوها از train هستند و منحنی‌شان out-of-fold است. خط نقطه‌چین خاکستری آستانه‌ی دسته‌ی همان ویدئوست. در ویدئوهای تصادف، خط‌چین نارنجی `time_of_alert` و خط‌چین قرمز `time_of_event` است.

</div>

In [22]:
sample_ids = sorted(p.stem for p in (DRIVE_DIR / "sample_videos").glob("*.mp4"))
examples = train[train["video_id"].isin(sample_ids)].sort_values(["label", "video_id"])
print("Sample videos found in train:", len(examples))

titles = [f"{'collision' if v.label else 'normal'} {v.video_id}" for v in examples.itertuples()]
fig = make_subplots(rows=2, cols=4, shared_yaxes=True, subplot_titles=titles)
for i, v in enumerate(examples.itertuples()):
    row, col = i // 4 + 1, i % 4 + 1
    c = curves[curves["video_path"] == v.video_path]
    fig.add_trace(go.Scatter(x=c["t_end"], y=c["score"], line_color="steelblue",
                             showlegend=False), row=row, col=col)
    fig.add_hline(y=THRESHOLD[v.fold], line_dash="dot", line_color="gray", row=row, col=col)
    if v.label:
        fig.add_vline(x=v.time_of_alert, line_dash="dash", line_color="orange", row=row, col=col)
        fig.add_vline(x=v.time_of_event, line_dash="dash", line_color="red", row=row, col=col)
fig.update_yaxes(range=[0, 1])
fig.update_xaxes(title_text="time (s)", row=2)
fig.update_layout(height=550, title="Out-of-fold risk curves of the dashboard sample videos")
fig.show()

Sample videos found in train: 8


<div dir="rtl">

**نتیجه.**
- فقط در تصادف `00872` منحنی شکل مورد انتظار را دارد: تا نزدیک alert پایین است و بعد تا لحظه‌ی حادثه بالا می‌رود.
- در تصادف‌های `00227` و `00713` امتیاز از ابتدای ویدئو بالای آستانه است و نزدیک alert بیشتر نمی‌شود. در `00754` امتیاز چند ثانیه قبل از alert به آستانه می‌رسد و نزدیک آن می‌ماند.
- از ویدئوهای عادی، `01372` و `01875` همیشه زیر آستانه‌اند، `01849` یک جهش کوتاه حدود ثانیه‌ی ۲۰ دارد و `01071` در نیمه‌ی اول ویدئو بیشتر بالای آستانه است.
- هشت ویدئو نمونه‌ی کوچکی است، ولی بخش بعد همین الگو را روی همه‌ی ویدئوها نشان می‌دهد: سطح امتیاز بیشتر به خود ویدئو بستگی دارد تا به لحظه‌ی خطر.

</div>

## Model vs Human Alert

<div dir="rtl">

`time_of_alert` لحظه‌ای است که برچسب‌زن‌ها خطر را دیده‌اند. برای هر لحظه نسبت به آن، از ۸ ثانیه قبل تا حادثه، سهم ویدئوهای تصادفی که امتیازشان بالای آستانه است حساب می‌شود. زمان هر نقطه نسبت به alert به نزدیک‌ترین ۰.۲ ثانیه گرد می‌شود و فقط نقطه‌های قبل از حادثه به کار می‌روند.

همین سهم در رانندگی عادی پایه‌ی مقایسه است. بالای آستانه بودن ۱ ثانیه قبل از alert فقط وقتی یعنی «زودتر از انسان» که از این پایه، و از سهم همان ویدئوها در چند ثانیه قبل‌تر، بیشتر باشد.

</div>

In [23]:
before_event = pos_curves[pos_curves["t_end"] < pos_curves["time_of_event"]].copy()
before_event["from_alert"] = (((before_event["t_end"] - before_event["time_of_alert"]) / 0.2)
                              .round() * 0.2).round(1)
per_video = before_event.drop_duplicates(["video_path", "from_alert"])
alert_share = per_video.groupby("from_alert")["above"].agg(share="mean", videos="size")
alert_share = alert_share.loc[-8:3].reset_index()
normal_share = footage["normal driving"]["above"].mean()

for name, t in [("At the alert", 0.0), ("1 s before the alert", -1.0)]:
    row = alert_share[alert_share["from_alert"] == t].iloc[0]
    print(f"{name}: {row['share']:.1%} of {row['videos']:.0f} collision videos above the threshold")
early = per_video[per_video["from_alert"].between(-8, -3)]
print(f"8 to 3 s before the alert: {early['above'].mean():.1%} of points above the threshold")
print(f"Normal driving: {normal_share:.1%} of points above the threshold")

fig = px.line(alert_share, x="from_alert", y="share", markers=True, hover_data=["videos"],
              labels={"from_alert": "seconds from the human alert",
                      "share": "collision videos above the threshold"},
              title="Collision videos above the threshold, aligned on the human alert")
fig.add_hline(y=normal_share, line_dash="dot", annotation_text="normal driving")
fig.add_vline(x=0, line_dash="dash", line_color="orange", annotation_text="alert")
fig.add_vline(x=train["gap"].median(), line_dash="dash", line_color="red",
              annotation_text="median event")
fig.update_layout(height=400, yaxis_range=[0, 1])
fig.show()

At the alert: 72.9% of 701 collision videos above the threshold
1 s before the alert: 69.5% of 709 collision videos above the threshold
8 to 3 s before the alert: 64.2% of points above the threshold
Normal driving: 38.4% of points above the threshold


<div dir="rtl">

**نتیجه.**
- ۸ تا ۳ ثانیه قبل از alert، ۶۴.۲٪ نقطه‌های ویدئوهای تصادف بالای آستانه‌اند، در برابر ۳۸.۴٪ در رانندگی عادی. پس بیشتر فرق دو دسته چند ثانیه قبل از خطر هم وجود دارد و به خود ویدئو مربوط است.
- در نمودار، سهم از حدود ۲.۵ ثانیه قبل از alert کمی بالا می‌رود: ۶۹.۵٪ در ۱ ثانیه قبل و ۷۲.۹٪ در لحظه‌ی alert. این از هر دو پایه بیشتر است، ولی ۱ ثانیه قبل از alert فقط ۵.۳ واحد درصد از ۸ تا ۳ ثانیه قبلِ همان ویدئوها بالاتر است. `time_of_alert` هم دومین زمانِ دیرتر بین ۱۰ برچسب‌زن است، پس مقایسه با یک برچسب‌زن دیرتر از بقیه است.
- بعد از alert افزایش تندتر است و نزدیک حادثه از ۸۰٪ می‌گذرد. در این بخش ویدئوهایی که حادثه‌شان زودتر رخ داده کنار می‌روند، پس ترکیب ویدئوها در هر نقطه یکی نیست.

</div>

## Conditions

<div dir="rtl">

برای هر گروه نور، آب‌وهوا و صحنه، در ۱ ثانیه قبل از حادثه دو سهم حساب می‌شود: سهم تصادف‌های بالای آستانه (recall) و سهم پنجره‌های عادی بالای آستانه. اگر یک گروه هر دو را بالا داشته باشد، مدل بیشتر خودِ شرایط را دیده تا خطر را. بازه‌ها Wilson هستند. گروه‌هایی که کمتر از ۱۰ ویدئو دارند کنار می‌روند.

</div>

In [24]:
CONDITIONS = ["light_conditions", "weather", "scene"]
at_1s = evaluation[evaluation["delta"] == 1.0].merge(train[["video_path"] + CONDITIONS],
                                                     on="video_path")
at_1s["above"] = at_1s["score"] >= at_1s["fold"].map(THRESHOLD)
rows = []
for column in CONDITIONS:
    for (group, label), g in at_1s.groupby([column, "label"]):
        if len(g) < 10:
            continue
        low, high = proportion_confint(g["above"].sum(), len(g), method="wilson")
        rows.append({"condition": column, "group": group,
                     "videos": "collision (recall)" if label else "normal driving",
                     "n": len(g), "above threshold": g["above"].mean(), "low": low, "high": high})
conditions = pd.DataFrame(rows)
conditions.to_csv(RESULT_DIR / "conditions.csv", index=False)

fig = make_subplots(rows=1, cols=3, shared_yaxes=True, subplot_titles=CONDITIONS)
for i, column in enumerate(CONDITIONS, start=1):
    for kind, color in [("collision (recall)", "firebrick"), ("normal driving", "steelblue")]:
        part = conditions[(conditions["condition"] == column) & (conditions["videos"] == kind)]
        fig.add_trace(go.Scatter(
            x=part["group"], y=part["above threshold"], mode="markers", name=kind,
            legendgroup=kind, showlegend=i == 1, marker=dict(color=color, size=10),
            customdata=part["n"], hovertemplate="%{x}: %{y:.2f} (n=%{customdata})",
            error_y=dict(type="data", array=part["high"] - part["above threshold"],
                         arrayminus=part["above threshold"] - part["low"])), row=1, col=i)
fig.update_yaxes(range=[0, 1])
fig.update_layout(height=420, title="Share above the threshold 1 s before the event, by condition")
fig.show()
conditions.round(3)

,condition,group,videos,n,above threshold,low,high
0,light_conditions,Dark,normal driving,19,0.053,0.009,0.246
1,light_conditions,Dark,collision (recall),17,0.529,0.310,0.738
2,light_conditions,Normal,normal driving,496,0.393,0.351,0.437
3,light_conditions,Normal,collision (recall),499,0.806,0.769,0.838
4,light_conditions,Twilight,normal driving,36,0.306,0.180,0.469
5,light_conditions,Twilight,collision (recall),34,0.824,0.665,0.917
6,weather,Clear,normal driving,368,0.348,0.301,0.398
7,weather,Clear,collision (recall),331,0.785,0.738,0.826
8,weather,Cloudy,normal driving,169,0.396,0.326,0.472
9,weather,Cloudy,collision (recall),178,0.787,0.721,0.840


<div dir="rtl">

**نتیجه.**
- آستانه طوری است که recall کل ۸۰٪ باشد، پس recall هر گروه نسبی است و دو سهم با هم خوانده می‌شوند.
- در Urban هر دو سهم بالاست (عادی ۰.۵۰۶، تصادف ۰.۸۵۸) و در Sub-urban هر دو پایین (۰.۱۴۴ و ۰.۵۲۴). Rain (۰.۸۲۴ و ۰.۹۵۶، با ۱۷ ویدئوی عادی) و Dark (۰.۰۵۳ و ۰.۵۲۹، با ۱۹ و ۱۷ ویدئو) هم همین الگو را دارند، ولی بازه‌هایشان پهن است.
- پس سطح امتیاز با صحنه و شرایط جابه‌جا می‌شود و مدل بخشی از خودِ شرایط را می‌بیند، نه فقط خطر را.
- Highway (۰.۴۰۱ و ۰.۷۷۳) و نور Normal (۰.۳۹۳ و ۰.۸۰۶) نزدیک کل داده‌اند. گروه Other فقط ویدئوی تصادف کافی داشت (۱۸).

</div>

## Time of the Event

<div dir="rtl">

منحنی ویدئوهای تصادف تا ۲ ثانیه بعد از حادثه ادامه دارد. لحظه‌ای که منحنی بیشترین مقدار را دارد، به‌عنوان تخمین لحظه‌ی حادثه با `time_of_event` مقایسه می‌شود.

ویدئوها حول حادثه بریده شده‌اند، پس یک پایه‌ی ساده هم سنجیده می‌شود: «همیشه میانه‌ی زمان حادثه». این میانه برای هر ویدئو از ویدئوهای تصادفِ دسته‌های دیگر حساب می‌شود. مدل فقط وقتی لحظه‌ی حادثه را می‌شناسد که از این پایه بهتر باشد.

</div>

In [25]:
collision_curves = curves[curves["label"] == 1]
peaks = collision_curves.loc[collision_curves.groupby("video_path")["score"].idxmax(),
                             ["video_path", "fold", "t_end"]]
peaks = peaks.merge(train[["video_path", "time_of_event"]], on="video_path")
collisions = train[train["label"] == 1]
other_folds = {k: collisions.loc[collisions["fold"] != k, "time_of_event"].median()
               for k in range(N_FOLDS)}
errors = {"model (curve peak)": peaks["t_end"] - peaks["time_of_event"],
          "baseline (median event time)": peaks["fold"].map(other_folds) - peaks["time_of_event"]}

rows = []
for name, e in errors.items():
    rows.append({"estimate": name, "median |error| (s)": e.abs().median(),
                 "within 0.5 s": (e.abs() <= 0.5).mean(), "within 1 s": (e.abs() <= 1).mean(),
                 "median error (s)": e.median(),
                 "beyond 10 s": (e.abs() > 10).mean()})
timing = pd.DataFrame(rows)
print("Collision videos:", len(peaks))

fig = go.Figure([go.Histogram(x=e.clip(-10, 10), name=name, opacity=0.6,
                              xbins=dict(start=-10, end=10.2, size=0.2))
                 for name, e in errors.items()])
fig.add_vline(x=0, line_dash="dash")
fig.update_layout(barmode="overlay", height=400, xaxis_title="estimate - time_of_event (s)",
                  yaxis_title="videos",
                  title="Error of the estimated event time (errors beyond ±10 s at the edges)")
fig.show()
timing.round(3)

Collision videos: 750


,estimate,median |error| (s),within 0.5 s,within 1 s,median error (s),beyond 10 s
0,model (curve peak),1.467,0.269,0.407,-0.433,0.165
1,baseline (median event time),0.600,0.424,0.715,0.000,0.039


<div dir="rtl">

**نتیجه.**
- مدل از پایه بدتر است: میانه‌ی قدر مطلق خطا ۱.۴۶۷ ثانیه در برابر ۰.۶۰۰، و ۴۰.۷٪ تخمین‌ها در فاصله‌ی ۱ ثانیه از حادثه‌اند در برابر ۷۱.۵٪ برای پایه.
- بیشینه‌ی منحنی در ۱۶.۵٪ ویدئوها بیش از ۱۰ ثانیه با حادثه فاصله دارد (پایه: ۳.۹٪). این ویدئوها در لبه‌ی چپ نمودارند، یعنی منحنی خیلی زودتر از حادثه به بیشترین مقدارش رسیده است.
- پس مدل لحظه‌ی حادثه را نمی‌شناسد. منحنی بعد از `event − 0.3` برچسب آموزشی هم نداشته است. این با بخش‌های قبل یکی است: امتیاز بالا بیشتر به کل ویدئو مربوط است تا به لحظه‌ی خطر.

</div>

## Training Length

<div dir="rtl">

شبکه‌ها در بیشتر دسته‌ها بعد از ۱ epoch بهترین بودند. برای دیدن دلیلش، شبکه‌ی انتخاب‌شده در هر دسته ۱۰ epoch بدون توقف زودهنگام آموزش می‌بیند، با همان پنجره‌ها و همان تنظیمات بخش Models. بعد از هر epoch سه عدد ثبت می‌شود: AP روی ویدئوهای توقف زودهنگام، mAP روی دسته‌ی کنار گذاشته و AUC داخل یک ویدئو.

این بخش فقط توصیفی است و تعداد epoch مدل نهایی را عوض نمی‌کند.

</div>

In [26]:
def epoch_curve(k, net_key, n_epochs=10):
    # same rows and training loop as run_fold and fit_net, without early stopping
    fold_videos = train[train["fold"] != k]
    groups = fold_videos.drop_duplicates("group")
    inner_groups = groups.groupby("label").sample(frac=INNER_SHARE, random_state=SEED + k)["group"]
    inner = set(fold_videos.loc[fold_videos["group"].isin(inner_groups), "video_path"])
    fit_rows = windows[(windows["fold"] != k) & ~windows["video_path"].isin(inner)]
    fit_rows = fit_rows[frames_before(fit_rows) >= FINAL_LENGTH]
    val_rows = eval_windows[eval_windows["video_path"].isin(inner)]
    held = final_scores[final_scores["fold"] == k].drop(columns="score")

    P = project(joblib.load(CACHE_DIR / f"pca_fold{k}.joblib"))
    torch.manual_seed(SEED)
    order_rng = np.random.default_rng(SEED)
    model = (GRUNet if net_key == "gru" else ConvNet)().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=NET_LR, weight_decay=1e-2)
    end, y = fit_rows["end"].to_numpy(), fit_rows["label"].to_numpy()
    rows = []
    for epoch in range(n_epochs):
        model.train()
        order = order_rng.permutation(len(end))
        for i in range(0, len(end), 256):
            batch = order[i:i + 256]
            x = torch.from_numpy(P[end[batch, None] + offsets(FINAL_LENGTH)]).to(device)
            target = torch.from_numpy(y[batch].astype(np.float32)).to(device)
            loss = F.binary_cross_entropy_with_logits(model(x), target)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        scored = held.assign(score=predict_net(model, P, held["end"].to_numpy(), FINAL_LENGTH))
        val_score = predict_net(model, P, val_rows["end"].to_numpy(), FINAL_LENGTH)
        rows.append({"fold": k, "epoch": epoch + 1,
                     "early-stopping AP": average_precision_score(val_rows["label"], val_score),
                     "held-out mAP": mean_ap(scored[scored["set"] == "evaluation"]),
                     "same-video AUC": same_video_auc(scored[scored["set"] == "same video"])})
    return rows


started = time.time()
epochs = pd.DataFrame([row for k in range(N_FOLDS) for row in epoch_curve(k, best_net)])
epochs.to_csv(RESULT_DIR / "epoch_curve.csv", index=False)
best = epochs.loc[epochs.groupby("fold")["early-stopping AP"].idxmax(), "epoch"]
print(f"{MODELS[best_net]} | epoch with the highest early-stopping AP in each fold: "
      f"{best.tolist()} | {time.time() - started:.0f}s")
print("\nEarly-stopping AP after each epoch (columns: fold):")
print(epochs.pivot(index="epoch", columns="fold", values="early-stopping AP").round(3))

mean_curve = epochs.drop(columns="fold").groupby("epoch").mean().reset_index()
fig = px.line(mean_curve.melt(id_vars="epoch", var_name="metric", value_name="value"),
              x="epoch", y="value", color="metric", markers=True,
              title=f"{MODELS[best_net]}: scores after each epoch (mean of {N_FOLDS} folds)")
fig.update_layout(height=400)
fig.show()
mean_curve.round(3)

1D CNN | epoch with the highest early-stopping AP in each fold: [1, 1, 1, 10, 10] | 1284s

Early-stopping AP after each epoch (columns: fold):
fold       0      1      2      3      4
epoch                                   
1      0.748  0.691  0.743  0.749  0.682
2      0.724  0.651  0.723  0.766  0.741
3      0.696  0.691  0.704  0.763  0.722
4      0.715  0.669  0.703  0.750  0.673
5      0.682  0.678  0.701  0.759  0.716
6      0.689  0.689  0.690  0.770  0.720
7      0.709  0.642  0.689  0.759  0.715
8      0.716  0.655  0.678  0.770  0.701
9      0.683  0.663  0.661  0.775  0.720
10     0.684  0.660  0.697  0.776  0.743


,epoch,early-stopping AP,held-out mAP,same-video AUC
0,1,0.723,0.739,0.663
1,2,0.721,0.726,0.689
2,3,0.715,0.717,0.703
3,4,0.702,0.716,0.708
4,5,0.707,0.708,0.711
5,6,0.712,0.699,0.705
6,7,0.703,0.693,0.706
7,8,0.704,0.685,0.706
8,9,0.700,0.683,0.707
9,10,0.712,0.688,0.705


<div dir="rtl">

**نتیجه.**
- بهترین epoch روی ویدئوهای توقف زودهنگام در دسته‌ها ۱، ۱، ۱، ۱۰ و ۱۰ است. در دسته‌ی پنجم، بخش Models epoch دوم را انتخاب کرده بود، چون آموزش بعد از ۵ epoch بدون بهتر شدن متوقف می‌شد. اینجا epoch دهم (۰.۷۴۳) کمی از epoch دوم (۰.۷۴۱) بهتر شد.
- با آموزش بیشتر، میانگین mAP دسته‌ی کنار گذاشته تقریباً پیوسته کم می‌شود: ۰.۷۳۹ بعد از epoch اول و ۰.۶۸۸ بعد از epoch دهم.
- در همان حال AUC داخل یک ویدئو از ۰.۶۶۳ به ۰.۷۱۱ (epoch پنجم) می‌رسد. پس آموزش بیشتر پنجره‌های خطر را از منفی‌های سختِ همان ویدئو بهتر جدا می‌کند، ولی mAP، که پنجره‌های ویدئوهای مختلف را با هم مقایسه می‌کند، بدتر می‌شود.
- این نتیجه با ۱ epoch مدل نهایی سازگار است. عددهای این جدول میانگین پنج دسته‌اند و با جدول مقایسه، که امتیاز همه‌ی دسته‌ها را کنار هم می‌گذارد، مستقیم مقایسه نمی‌شوند.

</div>

## Final Model

<div dir="rtl">

مدل نهایی همان ترکیب «LightGBM، فریم آخر» و 1D CNN با پنجره‌ی ۲ ثانیه است، این بار روی همه‌ی پنجره‌های train. PCA هم روی فریم‌های همه‌ی پنجره‌های train ساخته می‌شود. داده‌ای برای توقف زودهنگام کنار گذاشته نمی‌شود، پس تعداد درخت‌ها و epochها میانه‌ی پنج دسته در cross-validation است. یک مدل فقط‌متادیتا هم با همان ۱۰۰ درخت آموزش می‌بیند، تا روی آزمون هم معلوم باشد بدون دیدن تصویر چه امتیازی به دست می‌آید.

آستانه‌ی هشدار قبل از دیدن آزمون ثابت می‌شود: میانه‌ی آستانه‌ی پنج دسته. مدل‌ها، PCA و تنظیمات برای داشبورد ذخیره می‌شوند. آزمون در هیچ‌کدام از این‌ها به کار نمی‌رود.

</div>

In [27]:
import json

MODEL_DIR = DRIVE_DIR / "models"
MODEL_DIR.mkdir(exist_ok=True)
final_rows = windows[frames_before(windows) >= FINAL_LENGTH]
final_end, final_y = final_rows["end"].to_numpy(), final_rows["label"].to_numpy()
final_sizes = {key: int(load_scores([key], FINAL_LENGTH).groupby("fold")["size"].first().median())
               for key in FINAL_PARTS}

pca = fit_pca(final_end)
P = project(pca)
final_models = {}
for key in FINAL_PARTS:
    if key in TREES:
        X = summarize(P, final_end, key, FINAL_LENGTH)
        final_models[key] = fit_tree(X, final_y, n_trees=final_sizes[key])
    else:
        net_class = GRUNet if key == "gru" else ConvNet
        final_models[key] = fit_net(net_class, P, final_end, final_y, None, None, FINAL_LENGTH,
                                    epochs=final_sizes[key])
meta_model = fit_tree(meta_features(final_rows), final_y, n_trees=100)
FINAL_THRESHOLD = THRESHOLD.median()

print(f"Training windows: {len(final_rows)} from {final_rows['video_path'].nunique()} videos | "
      f"PCA variance kept {pca.explained_variance_ratio_.sum():.1%}")
print("Trees or epochs:", {MODELS[key]: n for key, n in final_sizes.items()})
print(f"Threshold (median of the fold thresholds): {FINAL_THRESHOLD:.4f}")

joblib.dump(pca, MODEL_DIR / "final_pca.joblib")
for key, model in final_models.items():
    save_model(key, model, MODEL_DIR / f"final_{key}")
config = {"parts": FINAL_PARTS, "score": "mean of the parts' probabilities",
          "window_frames": int(FINAL_LENGTH), "fps": 10, "pca_dim": PCA_DIM,
          "sizes": final_sizes, "threshold": float(FINAL_THRESHOLD),
          "fold_thresholds": [float(t) for t in THRESHOLD],
          "encoder": "facebook/dinov2-small", "frame_size": [392, 224],
          "frame_vector": "CLS + mean patch of the 4 image quadrants, 5 x 384"}
(MODEL_DIR / "final_config.json").write_text(json.dumps(config, indent=2))
print("Saved:", sorted(p.name for p in MODEL_DIR.glob("final_*")))

Training windows: 18782 from 1500 videos | PCA variance kept 91.1%
Trees or epochs: {'LightGBM, last frame': 69, '1D CNN': 1}
Threshold (median of the fold thresholds): 0.1989
Saved: ['final_cnn.pt', 'final_config.json', 'final_last.joblib', 'final_pca.joblib']


<div dir="rtl">

**نتیجه.**
- مدل نهایی روی ۱۸٬۷۸۲ پنجره از ۱۵۰۰ ویدئو آموزش دید: ۶۹ درخت برای فریم آخر و ۱ epoch برای CNN. PCA ۹۱.۱٪ واریانس را نگه داشت.
- آستانه‌ی مدل نهایی، قبل از دیدن آزمون، ۰.۱۹۸۹ است.

</div>

## Test Clips

<div dir="rtl">

آزمون رسمی ۱۳۴۴ کلیپ حدوداً ۱۰ ثانیه‌ای دارد که Δ ثانیه (۰.۵، ۱ یا ۱.۵) قبل از حادثه تمام می‌شوند. مدل فقط ۲۰ فریم آخر هر کلیپ را می‌بیند. چون پنجره همیشه آخر کلیپ است، طول کلیپ روی امتیاز اثری ندارد.

هر ویدئوی منبع تا سه کلیپ دارد، یکی برای هر Δ، و `time_to_accident_test_map.csv` می‌گوید کدام کلیپ‌ها از یک ویدئو بریده شده‌اند. کلیپ‌های یک ویدئو به هم وابسته‌اند، پس تعداد ویدئوهای منبع چاپ می‌شود و بازه‌های بخش بعد با bootstrap روی ویدئوهای منبع ساخته می‌شوند. طول کلیپ‌ها و سهم ویدئوهای ۳۰ fps هم برای هر برچسب چاپ می‌شود.

</div>

In [28]:
test_clips = videos[videos["split"] != "train"].copy()
test_clips["delta"] = test_clips["time_to_accident"]
test_clips["end"] = test_clips["start"] + test_clips["frames"] - 1
source_of = {int(x): f"source {i}" for i, ids in enumerate(test_map.itertuples(index=False))
             for x in ids if pd.notna(x)}
test_clips["source"] = [source_of.get(int(v), f"clip {v}") for v in test_clips["video_id"]]
test_clips["30 fps"] = test_clips["source_fps"] == 30

print(f"Clips with fewer than {FINAL_LENGTH} frames:", (test_clips["frames"] < FINAL_LENGTH).sum())
print("Clips not in the test map:", test_clips["source"].str.startswith("clip").sum())
print("Source videos in both subsets:", (test_clips.groupby("source")["split"].nunique() > 1).sum())
print("\nClips per group (columns: seconds before the event):")
print(pd.crosstab([test_clips["split"], test_clips["label"]], test_clips["delta"]))
facts = test_clips.groupby(["split", "label"]).agg(
    clips=("video_id", "size"), source_videos=("source", "nunique"),
    mean_length_s=("last_time", "mean"), share_30fps=("30 fps", "mean"))
print("\nTest clips by subset and label:")
print(facts.round(3))
train_30 = (train["source_fps"] == 30).groupby(train["label"]).mean()
print("\nShare of 30 fps videos in train (by label):", train_30.round(3).to_dict())

Clips with fewer than 20 frames: 0
Clips not in the test map: 0
Source videos in both subsets: 0

Clips per group (columns: seconds before the event):
delta               0.5  1.0  1.5
split        label               
test-private 0      142  116   81
             1      142  117   79
test-public  0      142  116   75
             1      142  115   77

Test clips by subset and label:
                    clips  source_videos  mean_length_s  share_30fps
split        label                                                  
test-private 0        339            142          9.753        0.637
             1        338            142          9.936        0.512
test-public  0        333            142          9.654        0.682
             1        334            142          9.875        0.551

Share of 30 fps videos in train (by label): {0: 0.589, 1: 0.6}


<div dir="rtl">

**نتیجه.**
- هر زیرمجموعه ۱۴۲ ویدئوی منبع تصادف و ۱۴۲ ویدئوی منبع عادی دارد، در حالی که هر برچسب حدود ۳۳۵ کلیپ دارد. پس عدم قطعیت آزمون بیشتر از چیزی است که تعداد کلیپ‌ها نشان می‌دهد. هیچ ویدئوی منبعی در هر دو زیرمجموعه نیست و همه‌ی کلیپ‌ها دست‌کم ۲۰ فریم دارند.
- کلیپ‌های عادی کمی کوتاه‌ترند (۹.۶۵ و ۹.۷۵ ثانیه در برابر ۹.۸۸ و ۹.۹۴ ثانیه)، ولی پنجره همیشه آخر کلیپ است و طول کلیپ به مدل نمی‌رسد.
- سهم ویدئوهای ۳۰ fps در آزمون برای کلیپ‌های عادی بیشتر است (۰.۶۳۷ و ۰.۶۸۲ در برابر ۰.۵۱۲ و ۰.۵۵۱)، در حالی که در train برای دو برچسب تقریباً یکی است (۰.۵۸۹ و ۰.۶۰۰).

</div>

## Official Test

<div dir="rtl">

مدل نهایی و مدل متادیتا به هر کلیپ آزمون یک امتیاز می‌دهند. mAP، AP هر Δ و AUC برای test-public و test-private جدا حساب می‌شوند و کنار عدد cross-validation همان مدل قرار می‌گیرند. ردیف test (all) همه‌ی ۱۳۴۴ کلیپ را با هم دارد، چون عددهای منتشرشده روی همین مجموعه‌اند.

فایل `submission.csv` با اسکریپت رسمی `evaluate_submission.py` هم سنجیده می‌شود، تا معلوم باشد mAP این نوت‌بوک همان معیار رسمی است. برچسب‌های آزمون فقط همین‌جا و بخش بعد به کار می‌روند و بعد از دیدن آن‌ها هیچ انتخابی عوض نمی‌شود.

</div>

In [29]:
test_clips["score"] = final_score(final_models, P, test_clips)
X_meta = (pd.get_dummies(test_clips[["light_conditions", "weather", "scene"]])
          .reindex(columns=meta.columns, fill_value=0).astype(np.float32).to_numpy())
test_clips["meta score"] = meta_model.predict_proba(X_meta)[:, 1]


def test_row(name, table):
    # 95% interval by resampling source videos, since clips of one video are related
    low, high = bootstrap_ci(table.assign(video_path=table["source"]))
    row = {"set": name, "mAP": mean_ap(table), "CI low": low, "CI high": high}
    for d, g in table.groupby("delta"):
        row[f"AP@{d}s"] = average_precision_score(g["label"], g["score"])
    row["AUC"] = roc_auc_score(table["label"], table["score"])
    return row


subsets = [("test-public", test_clips[test_clips["split"] == "test-public"]),
           ("test-private", test_clips[test_clips["split"] == "test-private"]),
           ("test (all)", test_clips)]
cv = summary.set_index("model")
metrics = ["mAP", "CI low", "CI high", "AP@0.5s", "AP@1.0s", "AP@1.5s", "AUC"]
rows = []
for key, column in [(FINAL, "score"), ("meta", "meta score")]:
    rows.append({"model": MODELS[key], "set": "cross-validation", **cv.loc[MODELS[key], metrics]})
    for name, part in subsets:
        rows.append({"model": MODELS[key], **test_row(name, part.assign(score=part[column]))})
test_results = pd.DataFrame(rows)
test_results.to_csv(RESULT_DIR / "test_results.csv", index=False)

fig = px.bar(test_results, x="set", y="mAP", color="model", barmode="group",
             error_y=test_results["CI high"] - test_results["mAP"],
             error_y_minus=test_results["mAP"] - test_results["CI low"],
             title="mAP in cross-validation and on the official test (95% interval)")
fig.add_hline(y=0.5, line_dash="dot", annotation_text="random")
fig.update_layout(height=420, yaxis_range=[0.4, 1])
fig.show()

submission = pd.DataFrame({"id": test_clips["video_id"].astype(int), "score": test_clips["score"]})
submission.to_csv(RESULT_DIR / "submission.csv", index=False)
for name, part in subsets[:2]:
    print(f"mAP ({name}) computed here: {mean_ap(part):.6f}")
print("Official script:")
!python3 "{DATA_DIR}/evaluate_submission.py" "{RESULT_DIR}/submission.csv" "{DATA_DIR}/solution.csv"
test_results.round(3)

mAP (test-public) computed here: 0.752987
mAP (test-private) computed here: 0.747544
Official script:
mAP (Public): 0.752987
mAP (Private): 0.747544


,model,set,mAP,CI low,CI high,AP@0.5s,AP@1.0s,AP@1.5s,AUC
0,"LightGBM, last frame + 1D CNN",cross-validation,0.731,0.692,0.769,0.726,0.733,0.734,0.753
1,"LightGBM, last frame + 1D CNN",test-public,0.753,0.678,0.825,0.749,0.745,0.765,0.756
2,"LightGBM, last frame + 1D CNN",test-private,0.748,0.665,0.837,0.765,0.741,0.737,0.781
3,"LightGBM, last frame + 1D CNN",test (all),0.746,0.690,0.802,0.754,0.741,0.744,0.767
4,"LightGBM, metadata only",cross-validation,0.576,0.535,0.619,0.579,0.578,0.569,0.604
5,"LightGBM, metadata only",test-public,0.587,0.512,0.670,0.602,0.586,0.572,0.605
6,"LightGBM, metadata only",test-private,0.583,0.500,0.671,0.585,0.590,0.574,0.595
7,"LightGBM, metadata only",test (all),0.582,0.526,0.638,0.592,0.585,0.569,0.599


<div dir="rtl">

**نتیجه.**
- اسکریپت رسمی همان عدد این نوت‌بوک را می‌دهد: mAP برابر ۰.۷۵۳ در test-public و ۰.۷۴۸ در test-private.
- روی هر ۱۳۴۴ کلیپ، mAP برابر ۰.۷۴۶ (بازه‌ی ۹۵٪: ۰.۶۹۰ تا ۰.۸۰۲) و AUC برابر ۰.۷۶۷ است. mAP آزمون کمی بیشتر از عدد cross-validation (۰.۷۳۱) و داخل بازه‌ی آن (۰.۶۹۲ تا ۰.۷۶۹) است، پس cross-validation خوش‌بینانه نبوده است.
- بازه‌ی هر زیرمجموعه پهن است (public: ۰.۶۷۸ تا ۰.۸۲۵، private: ۰.۶۶۵ تا ۰.۸۳۷) و فرق دو زیرمجموعه (۰.۰۰۵) در برابر آن ناچیز است.
- مدل متادیتا روی همه‌ی آزمون ۰.۵۸۲ می‌گیرد، نزدیک عدد cross-validation آن (۰.۵۷۶). یعنی روی آزمون هم بخشی از امتیاز بدون دیدن تصویر به دست می‌آید، و تصویر mAP را از ۰.۵۸۲ به ۰.۷۴۶ می‌رساند.
- AP در سه Δ نزدیک هم است (۰.۷۵۴، ۰.۷۴۱ و ۰.۷۴۴)، مثل cross-validation.

</div>

## Alarm Threshold on the Test Set

<div dir="rtl">

آستانه از مدل‌های پنج دسته آمده است، ولی مدل نهایی روی همه‌ی داده آموزش دیده و مقیاس امتیازش می‌تواند فرق کند. برای دیدن این اثر، همان دو سهم بخش Alarm Threshold روی کلیپ‌های آزمون با Δ برابر ۱ حساب می‌شود: سهم کلیپ‌های تصادفِ بالای آستانه (recall) و سهم کلیپ‌های عادیِ بالای آستانه. میانه‌ی امتیاز کلیپ‌های عادی هم کنار میانه‌ی پنجره‌های عادی cross-validation چاپ می‌شود. آستانه بعد از دیدن این عددها عوض نمی‌شود.

</div>

In [30]:
cv_1s = evaluation[evaluation["delta"] == 1.0]
cv_above = cv_1s["score"] >= cv_1s["fold"].map(THRESHOLD)
print(f"cross-validation, threshold of each fold: collisions at 1 s "
      f"{cv_above[cv_1s['label'] == 1].mean():.1%} | normal at 1 s "
      f"{cv_above[cv_1s['label'] == 0].mean():.1%} | median score of normal "
      f"{cv_1s.loc[cv_1s['label'] == 0, 'score'].median():.4f}")
test_1s = test_clips[test_clips["delta"] == 1.0]
for split, part in test_1s.groupby("split"):
    above = part["score"] >= FINAL_THRESHOLD
    print(f"{split}, threshold {FINAL_THRESHOLD:.4f}: collisions at 1 s "
          f"{above[part['label'] == 1].mean():.1%} | normal at 1 s "
          f"{above[part['label'] == 0].mean():.1%} | median score of normal "
          f"{part.loc[part['label'] == 0, 'score'].median():.4f}")

names = test_clips["label"].map({1: "collision", 0: "normal driving"})
fig = px.histogram(test_clips, x="score", color=names, barmode="overlay", nbins=50,
                   opacity=0.6, labels={"color": "clip"},
                   title="Final model scores on the official test clips")
fig.add_vline(x=FINAL_THRESHOLD, line_dash="dash", annotation_text="threshold")
fig.update_layout(height=400)
fig.show()

cross-validation, threshold of each fold: collisions at 1 s 79.8% | normal at 1 s 37.7% | median score of normal 0.1578
test-private, threshold 0.1989: collisions at 1 s 86.3% | normal at 1 s 54.3% | median score of normal 0.2119
test-public, threshold 0.1989: collisions at 1 s 89.6% | normal at 1 s 53.4% | median score of normal 0.2206


<div dir="rtl">

**نتیجه.**
- با آستانه‌ی ثابت ۰.۱۹۸۹، recall در ۱ ثانیه روی آزمون ۸۶.۳ و ۸۹.۶٪ است، نه ۸۰٪. سهم کلیپ‌های عادیِ بالای آستانه هم ۵۴.۳ و ۵۳.۴٪ است، در برابر ۳۷.۷٪ در cross-validation.
- میانه‌ی امتیاز کلیپ‌های عادی ۰.۲۱۱۹ و ۰.۲۲۰۶ است، در برابر ۰.۱۵۷۸ در cross-validation. هر دو سهم با هم بالا رفته‌اند، پس امتیازهای مدل نهایی روی آزمون کلاً بالاتر است و آستانه‌ی مدل‌های پنج دسته برای مدل نهایی منتقل نمی‌شود. علتش می‌تواند مقیاس مدل نهایی باشد، که روی داده‌ی بیشتری آموزش دیده، یا فرق کلیپ‌های آزمون، و این نوت‌بوک این دو را از هم جدا نمی‌کند.
- طبق قاعده، آستانه عوض نمی‌شود. برای یک سیستم هشدار واقعی، آستانه باید با خود مدل نهایی روی داده‌ای جدا تعیین شود.
- در نمودار، امتیاز کلیپ‌های تصادف و عادی هم‌پوشانی زیادی دارد و بیشترین امتیاز حدود ۰.۶۵ است.

</div>

## Comparison with Published Results

<div dir="rtl">

عددهای منتشرشده روی همین ۱۳۴۴ کلیپ آزمون:
- **BADAS-Open** (Nexar): مدل ویدئویی V-JEPA2 که end-to-end روی همین ۱۵۰۰ ویدئوی train آموزش دیده است. AP برابر ۰.۸۶ و AUC برابر ۰.۸۸.
- **سیستم صنعتی هشدار برخورد جلو (FCW):** AP برابر ۰.۵۸ و AUC برابر ۰.۶۴. مدل‌های دانشگاهی متن‌باز: AP بین ۰.۴۸ و ۰.۵۳.
- **VideoMAE-2 Large:** mAP برابر ۰.۸۰ و رتبه‌ی ۱۱ چالش Kaggle.
- **فایل نمونه‌ی رسمی `sample_submission.csv`:** طبق README دیتاست، mAP برابر ۰.۸۴ در public و ۰.۸۶ در private. منشأ امتیازهایش گفته نشده است.

AP مقاله‌ی BADAS ممکن است با mAP رسمی (میانگین سه گروه Δ) یکی نباشد. فرق اصلی این مدل‌ها با مدل این نوت‌بوک این است که یک شبکه‌ی ویدئویی را روی خود ویدئوها fine-tune می‌کنند، ولی اینجا DINOv2 یخ‌زده است و هر فریم را جدا می‌بیند.

منبع‌ها: Goldshmidt و همکاران، *BADAS*، arXiv:2510.14876. Moura و همکاران، *Nexar Dashcam Collision Prediction Dataset and Challenge*، arXiv:2503.03848. کارت مدل `jatinmehra/Accident-Detection-using-Dashcam` در Hugging Face.

</div>

<div dir="rtl">

**نتیجه.**
- روی همین ۱۳۴۴ کلیپ، مدل این نوت‌بوک mAP برابر ۰.۷۴۶ و AUC برابر ۰.۷۶۷ دارد. این از سیستم صنعتی FCW (AP ۰.۵۸ و AUC ۰.۶۴) و مدل‌های دانشگاهی متن‌باز (AP ۰.۴۸ تا ۰.۵۳) بیشتر است، و از VideoMAE-2 Large (mAP ۰.۸۰) و BADAS-Open (AP ۰.۸۶ و AUC ۰.۸۸) کمتر.
- فاصله با BADAS-Open حدود ۰.۱ است، با این احتیاط که تعریف AP مقاله ممکن است با mAP رسمی یکی نباشد. BADAS-Open هم روی همین ۱۵۰۰ ویدئو آموزش دیده است، پس این فاصله بیشتر به روش برمی‌گردد: fine-tune کردن یک مدل ویدئویی در برابر ویژگی‌های یخ‌زده‌ی فریم به فریم.

</div>

## Limitations

<div dir="rtl">

- **تکیه بر فرق بین ویدئوها.** مدل متادیتا بدون دیدن تصویر در cross-validation به mAP ۰.۵۷۶ رسید. ۸ تا ۳ ثانیه قبل از alert، ۶۴.۲٪ نقطه‌های ویدئوهای تصادف بالای آستانه‌اند، در برابر ۳۸.۴٪ در رانندگی عادی.
- **هشدار اشتباه.** با recall ۸۰٪ در ۱ ثانیه، ۶۱۵ هشدار اشتباه در هر ساعت رانندگی عادی.
- **ویژگی‌ها و تنظیمات.** DINOv2-small یخ‌زده است و هر فریم را جدا می‌بیند. PCA (۲۵۶ بعد)، اندازه‌ی شبکه‌ها و تنظیمات LightGBM تنظیم نشده‌اند. فقط طول پنجره با قاعده‌ی از پیش ثابت سنجیده شد.
- **نوسان و انتخاب مدل.** تعداد درخت‌ها بین دسته‌ها ۵۰ تا ۳۱۷ است و توقف زودهنگام روی ۳۶۹ تا ۳۸۷ پنجره انجام شد. یک اجرای قبلی روی سخت‌افزار دیگر، که در این نوت‌بوک نیست، حدود ۰.۰۱ mAP فرق داشت. فاصله‌ی پنج مدل اول از پهنای بازه‌ها کمتر است، پس رتبه‌شان قابل اتکا نیست و انتخاب بهترین مدل کمی خوش‌بینانه است.
- **lr و epoch.** lr شبکه‌ها یک بار، بعد از یک اجرای اولیه که در این نوت‌بوک نیست، از 1e-3 به 3e-4 تغییر کرد. قاعده از قبل ثابت شد: 3e-4 بماند، چه نتیجه بهتر شود چه بدتر. شبکه‌ی نهایی ۱ epoch آموزش می‌بیند، و آموزش بیشتر mAP را پایین آورد (۰.۷۳۹ به ۰.۶۸۸).
- **برچسب‌ها.** طبق README دیتاست، تصادف و نزدیک به تصادف هر دو برچسب ۱ دارند. ۲۴ ویدئوی تصادف پنجره‌ی خطر ندارند. `time_of_alert` دومین زمانِ دیرتر بین ۱۰ برچسب‌زن است.
- **گروه‌های کوچک.** در ۱ ثانیه قبل از حادثه، Dark ۱۹ ویدئوی عادی و ۱۷ تصادف دارد و Rain ۱۷ ویدئوی عادی. گروه‌های زیر ۱۰ ویدئو کنار رفتند.
- **نرخ فریم در آزمون.** سهم ویدئوهای ۳۰ fps در train برای دو برچسب تقریباً یکی است (۰.۵۸۹ و ۰.۶۰۰)، ولی در آزمون برای کلیپ‌های عادی بیشتر است (۰.۶۳۷ و ۰.۶۸۲ در برابر ۰.۵۱۲ و ۰.۵۵۱). این فرق در train نیست و مدل نمی‌تواند از آن یاد بگیرد.
- **آزمون کوچک‌تر از ظاهرش است.** هر زیرمجموعه ۱۴۲ ویدئوی منبع تصادف و ۱۴۲ ویدئوی منبع عادی دارد و بازه‌ی mAP هر زیرمجموعه ۰.۱۵ تا ۰.۱۷ پهنا دارد.
- **آستانه منتقل نمی‌شود.** آستانه‌ی پنج دسته روی مدل نهایی recall آزمون در ۱ ثانیه را ۸۶ تا ۹۰٪ و سهم کلیپ‌های عادیِ بالای آستانه را حدود ۵۴٪ کرد، در برابر ۸۰٪ و ۳۷.۷٪ در cross-validation.

</div>

## Summary

<div dir="rtl">

- **مسئله:** هشدار پیش از تصادف روی دیتاست Nexar. مدل در هر لحظه فقط ۲ ثانیه‌ی آخر ویدئو را می‌بیند.
- **روش:** ویژگی‌های یخ‌زده‌ی DINOv2-small برای هر فریم، پنجره‌های خطر و منفی‌های سخت از خود ویدئوهای تصادف، و انتخاب مدل با cross-validation گروهی روی ویدئوها. مدل نهایی ترکیب LightGBM روی فریم آخر و یک 1D CNN روی ۲۰ فریم است.
- **آزمون رسمی:** mAP برابر ۰.۷۵۳ در public و ۰.۷۴۸ در private، نزدیک عدد cross-validation (۰.۷۳۱). مدل متادیتا بدون دیدن تصویر ۰.۵۸۲ می‌گیرد.
- **محدودیت اصلی:** امتیاز بیشتر به خود ویدئو و صحنه بستگی دارد تا به لحظه‌ی خطر، و با recall ۸۰٪ در رانندگی عادی ۶۱۵ هشدار اشتباه در ساعت داده می‌شود. مدل‌های ویدئوییِ fine-tune شده روی همین داده به ۰.۸۰ تا ۰.۸۶ رسیده‌اند.

</div>